# 03 - Real draft-and-verify loop (8B target, 1B draft, batch size 1)

**Generated by `scripts/build_kaggle_notebooks.py` (git=a253e1ed509c-dirty src+artifacts_sha256=0f135c92b058). Do not edit; change `src/lgmtp/` and rebuild.**

Embedded: the dev-selected parameters of every policy (`results/phase1/selected_params.json`), the causal morpheme prefix model (`results/phase1/prefix_model.pkl.gz`), and the fixed list of 100 devtest ids (`results/phase1/real_loop_ids.json`).

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**, Secret **`HF_TOKEN`**.
**Run:** *Save Version > Save & Run All (Commit)*. After the first 5 sentences the log prints `ESTIMATE: ...`; if it is too long, stop, lower `N_SENT` / `REPEATS` in CONFIG and rerun.
**Output:** `/kaggle/working/real_loop.zip`, to be unzipped into `data/real_loop/` in the repo. If a run dies, attach its output as input and set `RESUME_FROM`.

In [ ]:
!pip -q install -U "transformers==5.22.1" accelerate datasets huggingface_hub

In [ ]:
# === lgmtp source, generated by scripts/build_kaggle_notebooks.py: DO NOT EDIT ===
import os, sys, importlib
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
PKG = '/kaggle/working/lgmtp' if os.path.isdir('/kaggle/working') else os.path.abspath('lgmtp')
os.makedirs(PKG, exist_ok=True)
LGMTP_SOURCES = {
    '__init__': '"""lgmtp: linguistically guided multi-token prediction (CS769)."""\n',
    'boundaries': '"""Token bytes, Devanagari akshara units, word boundaries and per-token cut labels.\n\nExtracted from ``notebooks/legacy/test1_hindi_boundaries.ipynb`` (cell 2). The logic of the\nlegacy functions is unchanged; only type hints / docstrings were added, plus ``token_labels``.\n\nFixes bug B4 (never decode single byte-level tokens): everything works on token *bytes* and\nbyte offsets, and text is only decoded for whole sequences.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom itertools import accumulate\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\n# --------------------------------------------------------------------------- #\n# token -> bytes\n# --------------------------------------------------------------------------- #\n\n\ndef _bytes_to_unicode() -> dict[int, str]:\n    """GPT-2 byte<->unicode table (used by Llama-3 and Qwen byte-level BPE)."""\n    bs = (list(range(ord("!"), ord("~") + 1))\n          + list(range(ord("¡"), ord("¬") + 1))\n          + list(range(ord("®"), ord("ÿ") + 1)))\n    cs = bs[:]\n    n = 0\n    for b in range(256):\n        if b not in bs:\n            bs.append(b)\n            cs.append(256 + n)\n            n += 1\n    return dict(zip(bs, [chr(c) for c in cs]))\n\n\nBYTE_DECODER: dict[str, int] = {v: k for k, v in _bytes_to_unicode().items()}\n_HEX_RE = re.compile(r"^<0x([0-9A-Fa-f]{2})>$")\n\n\ndef token_to_bytes(tok: str) -> bytes:\n    """Raw bytes of a vocabulary token string (as returned by ``convert_ids_to_tokens``)."""\n    m = _HEX_RE.match(tok)\n    if m:  # SentencePiece byte-fallback token\n        return bytes([int(m.group(1), 16)])\n    if all(ch in BYTE_DECODER for ch in tok):  # GPT-2 style byte-level BPE\n        return bytes(BYTE_DECODER[ch] for ch in tok)\n    return tok.replace("▁", " ").encode("utf-8")  # SentencePiece unicode piece\n\n\nclass HFTok:\n    """Thin wrapper: text -> list of token byte-strings."""\n\n    def __init__(self, name_or_tokenizer):\n        if isinstance(name_or_tokenizer, str):\n            from transformers import AutoTokenizer\n            self.name = name_or_tokenizer\n            self.tok = AutoTokenizer.from_pretrained(name_or_tokenizer)\n        else:\n            self.name = "custom"\n            self.tok = name_or_tokenizer\n\n    def to_bytes(self, text: str) -> list[bytes]:\n        ids = self.tok(text, add_special_tokens=False)["input_ids"]\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(ids)]\n\n    def ids_to_bytes(self, ids: Sequence[int]) -> list[bytes]:\n        """Byte-strings for a list of token ids (no decoding of individual tokens)."""\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(list(ids))]\n\n\nQWEN2_PRETOK_REGEX = (r"(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}"\n                      r"| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+")\n\n\ndef legacy_qwen2_pretok_tokenizer(name: str = "Qwen/Qwen3.5-0.8B"):\n    """``name``\'s tokenizer.json with its split regex replaced by the older Qwen2 one (``\\\\p{L}+``,\n    no ``\\\\p{M}``), which splits before every combining mark.\n\n    Only used to explain the legacy Test 1 Qwen number (3.33 tokens/word): the legacy run loaded\n    the tokenizer this way. The model\'s real tokenizer (``[\\\\p{L}\\\\p{M}]+``) gives ~2.16.\n    """\n    from huggingface_hub import hf_hub_download\n    from tokenizers import Tokenizer\n    from transformers import PreTrainedTokenizerFast\n    j = json.load(open(hf_hub_download(name, "tokenizer.json"), encoding="utf-8"))\n    j["pre_tokenizer"]["pretokenizers"][0]["pattern"]["Regex"] = QWEN2_PRETOK_REGEX\n    return PreTrainedTokenizerFast(tokenizer_object=Tokenizer.from_str(json.dumps(j)))\n\n\n# --------------------------------------------------------------------------- #\n# akshara + word boundaries\n# --------------------------------------------------------------------------- #\n\n_C = "क-हक़-य़ॸ-ॿ"      # consonants\n_VIRAMA = "्"\n_NUKTA = "़"\n_ZW = "\u200c\u200d"                                 # ZWNJ / ZWJ\n_IVOW = "ऄ-औॠॡॲ-ॷ"     # independent vowels\n_MATRA = "ऺऻा-ौॎॏॕ-ॗॢॣ"\n_MOD = "ऀ-ः॑-॔"                  # candrabindu, anusvara, visarga, stress marks\n\n_AKSHARA_RE = re.compile(\n    rf"[{_C}]{_NUKTA}?(?:{_VIRAMA}[{_ZW}]?[{_C}]{_NUKTA}?)*(?:{_VIRAMA}[{_ZW}]?|[{_MATRA}]*)[{_MOD}]*"\n    rf"|[{_IVOW}]{_NUKTA}?[{_MATRA}]*[{_MOD}]*"\n)\n\n\ndef akshara_units(text: str) -> list[tuple[int, int]]:\n    """Return list of (start, end) char spans. Non-Devanagari chars are single units\n    with any following combining marks / ZW joiners attached."""\n    units, i, n = [], 0, len(text)\n    while i < n:\n        m = _AKSHARA_RE.match(text, i)\n        if m and m.end() > i:\n            j = m.end()\n        else:\n            j = i + 1\n            while j < n and (unicodedata.category(text[j]).startswith("M") or text[j] in "\u200c\u200d"):\n                j += 1\n        units.append((i, j))\n        i = j\n    return units\n\n\ndef is_word_boundary(text: str, c: int) -> bool:\n    """True if char offset ``c`` is a word boundary (text edge, whitespace or punctuation)."""\n    n = len(text)\n    if c <= 0 or c >= n:\n        return True\n    a, b = text[c - 1], text[c]\n    if a.isspace() or b.isspace():\n        return True\n    if unicodedata.category(a).startswith("P") or unicodedata.category(b).startswith("P"):\n        return True\n    return False\n\n\nEND, WORD, MIDWORD, MIDAKS, MIDUTF8 = "end", "word_boundary", "mid_word", "mid_akshara", "mid_utf8"\nCUT_CATEGORIES = (END, WORD, MIDWORD, MIDAKS, MIDUTF8)\nTAG = {END: "E", WORD: "W", MIDWORD: "w", MIDAKS: "a", MIDUTF8: "u"}\n\n\ndef classify(b: int, total: int, byte2char: dict[int, int], text: str, aks: set[int]) -> str:\n    """Category of a cut at byte offset ``b``.\n\n    byte2char maps byte offsets that start a char (plus ``total``) to char offsets;\n    aks is the set of char offsets that are akshara-unit boundaries.\n    """\n    if b >= total:\n        return END\n    c = byte2char.get(b)\n    if c is None:\n        return MIDUTF8\n    if c not in aks:\n        return MIDAKS\n    return WORD if is_word_boundary(text, c) else MIDWORD\n\n\ndef sentence_features(tb: Sequence[bytes]) -> dict | None:\n    """tb: list of token byte-strings. Returns None if bytes are not valid UTF-8."""\n    full = b"".join(tb)\n    try:\n        text = full.decode("utf-8")\n    except UnicodeDecodeError:\n        return None\n    byte2char, pos = {}, 0\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        pos += len(ch.encode("utf-8"))\n    byte2char[pos] = len(text)\n    units = akshara_units(text)\n    aks = {0} | {e for _, e in units}\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n    total = len(full)\n    return {\n        "text": text,\n        "units": units,\n        "cut": [classify(e, total, byte2char, text, aks) for e in ends],\n        "start": [classify(s, total, byte2char, text, aks) for s in starts],\n    }\n\n\n# --------------------------------------------------------------------------- #\n# per-token gold labels (analysis only; never read by policies)\n# --------------------------------------------------------------------------- #\n\n\ndef _valid_utf8_prefix_len(full: bytes) -> int:\n    """Length of the longest prefix of ``full`` that is valid UTF-8, allowing only a\n    truncated character at the very end (e.g. generation stopped at MAX_NEW)."""\n    for cut in range(len(full), max(len(full) - 4, -1), -1):\n        try:\n            full[:cut].decode("utf-8")\n            return cut\n        except UnicodeDecodeError:\n            continue\n    raise ValueError("token bytes are not valid UTF-8 (invalid sequence before the end)")\n\n\ndef token_labels(token_bytes: Sequence[bytes],\n                 special: Sequence[bool] | None = None) -> dict[str, np.ndarray]:\n    """Gold labels for each token t of a target sequence (analysis only, never read by policies).\n\n    end_cut[t]      : category of the cut AFTER token t (END/WORD/MIDWORD/MIDAKS/MIDUTF8)\n    start_cut[t]    : category of the cut BEFORE token t\n    word_initial[t] : token t starts a new word (start_cut in {WORD, END})\n    pos_in_word[t]  : 0 for word-initial tokens, then 1, 2, ...\n    word_id[t]      : index of the word the token belongs to\n    char_span[t]    : (start_char, end_char) in the decoded text, shape (n, 2)\n\n    ``special[t]`` marks special/control tokens (e.g. ``<|eot_id|>``). They are stripped before\n    labelling and get ``END`` for both cuts, ``word_initial=True``, ``pos_in_word=0``, a fresh\n    ``word_id`` and an empty char span at the end of the text. If ``special`` is None, no token is\n    treated as special.\n\n    A trailing incomplete UTF-8 character (output truncated mid-character) is tolerated: cuts at\n    its start or inside it are ``MIDUTF8`` (what follows is unknown). Invalid bytes anywhere else\n    raise ``ValueError``.\n    """\n    n = len(token_bytes)\n    special = [False] * n if special is None else [bool(s) for s in special]\n    if len(special) != n:\n        raise ValueError("special must have the same length as token_bytes")\n    keep = [t for t in range(n) if not special[t]]\n    tb = [token_bytes[t] for t in keep]\n\n    full = b"".join(tb)\n    valid = _valid_utf8_prefix_len(full)\n    text = full[:valid].decode("utf-8")\n    full_total = len(full)\n\n    byte2char, pos = {}, 0\n    char_of_byte = np.zeros(full_total + 1, dtype=np.int64)  # char containing byte b\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        nb = len(ch.encode("utf-8"))\n        char_of_byte[pos:pos + nb] = ci\n        pos += nb\n    byte2char[pos] = len(text)\n    char_of_byte[pos:] = len(text)  # trailing partial char maps past the text\n    aks = {0} | {e for _, e in akshara_units(text)}\n\n    def cat(b: int) -> str:\n        if b >= full_total:\n            return END\n        if b >= valid:  # inside the trailing truncated char\n            return MIDUTF8\n        return classify(b, valid, byte2char, text, aks)\n\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n\n    end_cut = np.full(n, END, dtype=object)\n    start_cut = np.full(n, END, dtype=object)\n    char_span = np.full((n, 2), len(text), dtype=np.int64)\n    for j, t in enumerate(keep):\n        s, e = starts[j], ends[j]\n        end_cut[t] = cat(e)\n        start_cut[t] = cat(s)\n        cs = int(char_of_byte[min(s, full_total)])\n        ce = int(char_of_byte[e - 1]) + 1 if e > s else cs\n        char_span[t] = (cs, min(ce, len(text)) if e <= valid else len(text))\n\n    word_initial = np.array([sc in (WORD, END) for sc in start_cut], dtype=bool)\n    pos_in_word = np.zeros(n, dtype=np.int64)\n    word_id = np.zeros(n, dtype=np.int64)\n    wid = -1\n    for t in range(n):\n        if word_initial[t] or t == 0:\n            wid += 1\n            pos_in_word[t] = 0\n        else:\n            pos_in_word[t] = pos_in_word[t - 1] + 1\n        word_id[t] = wid\n    return dict(end_cut=end_cut, start_cut=start_cut, word_initial=word_initial,\n                pos_in_word=pos_in_word, word_id=word_id, char_span=char_span)\n\n\n# --------------------------------------------------------------------------- #\n# corpus analysis\n# --------------------------------------------------------------------------- #\n\n\ndef analyze(token_bytes_per_sentence: Iterable[Sequence[bytes]], ks: Sequence[int]):\n    """Corpus statistics of token cuts and of fixed-k draft windows. Returns (tot, per_k)."""\n    tot = Counter()\n    per_k = {k: Counter() for k in ks}\n    for tb in token_bytes_per_sentence:\n        f = sentence_features(tb)\n        if f is None:\n            tot["skipped_invalid_utf8"] += 1\n            continue\n        n = len(tb)\n        if n < 2:\n            tot["skipped_short"] += 1\n            continue\n        text, cut, start = f["text"], f["cut"], f["start"]\n        tot["sentences"] += 1\n        tot["tokens"] += n\n        tot["words"] += len(text.split())\n        tot["chars"] += len(text)\n        tot["aksharas"] += len(f["units"])\n        tot["starts_mid_utf8"] += sum(1 for s in start if s == MIDUTF8)\n        tot["starts_mid_akshara"] += sum(1 for s in start if s == MIDAKS)\n        for j in range(n - 1):  # internal cuts only (a token follows)\n            tot["cuts"] += 1\n            tot["cut_" + cut[j]] += 1\n\n        wb = [c in (WORD, END) for c in cut]\n        pw = [0] + list(accumulate(1 if x else 0 for x in wb))\n        last_wb, lw = [], -1\n        for j in range(n):\n            if wb[j]:\n                lw = j\n            last_wb.append(lw)\n        next_wb, nw = [0] * n, n - 1\n        for j in range(n - 1, -1, -1):\n            if wb[j]:\n                nw = j\n            next_wb[j] = nw\n\n        for k in ks:\n            c = per_k[k]\n            for s in range(0, n - k):\n                e = s + k - 1\n                cat = cut[e]\n                c["windows"] += 1\n                c[cat] += 1\n                done = pw[e + 1] - pw[s]\n                c["completed_words"] += done\n                if done == 0:\n                    c["zero_word"] += 1\n                if cat not in (WORD, END):\n                    c["mid_n"] += 1\n                    c["fwd_sum"] += next_wb[e] - e\n                    c["back_sum"] += e - last_wb[e]\n    return tot, per_k\n\n\ndef pct(a: float, b: float) -> float:\n    return 100.0 * a / b if b else float("nan")\n\n\ndef summarize(name: str, tot: Counter, per_k: dict, ks: Sequence[int]) -> dict:\n    """Turn ``analyze`` counters into the Test 1 summary dict."""\n    S = {\n        "tokenizer": name,\n        "sentences": tot["sentences"],\n        "tokens": tot["tokens"],\n        "tokens_per_word": tot["tokens"] / max(tot["words"], 1),\n        "tokens_per_akshara": tot["tokens"] / max(tot["aksharas"], 1),\n        "tokens_per_char": tot["tokens"] / max(tot["chars"], 1),\n        "pct_tokens_start_mid_akshara": pct(tot["starts_mid_akshara"], tot["tokens"]),\n        "pct_tokens_start_mid_utf8": pct(tot["starts_mid_utf8"], tot["tokens"]),\n        "pct_cuts_word_boundary": pct(tot["cut_" + WORD] + tot["cut_" + END], tot["cuts"]),\n        "pct_cuts_mid_word": pct(tot["cut_" + MIDWORD], tot["cuts"]),\n        "pct_cuts_mid_akshara": pct(tot["cut_" + MIDAKS], tot["cuts"]),\n        "pct_cuts_mid_utf8": pct(tot["cut_" + MIDUTF8], tot["cuts"]),\n        "skipped_invalid_utf8": tot["skipped_invalid_utf8"],\n        "per_k": [],\n    }\n    for k in ks:\n        c = per_k[k]\n        w = c["windows"]\n        S["per_k"].append({\n            "k": k,\n            "windows": w,\n            "pct_end_at_word_boundary": pct(c[WORD] + c[END], w),\n            "pct_end_mid_word": pct(c[MIDWORD], w),\n            "pct_end_mid_akshara": pct(c[MIDAKS], w),\n            "pct_end_mid_utf8": pct(c[MIDUTF8], w),\n            "pct_end_anywhere_inside_word": 100.0 - pct(c[WORD] + c[END], w) if w else float("nan"),\n            "mean_words_completed": c["completed_words"] / w if w else float("nan"),\n            "pct_drafts_completing_zero_words": pct(c["zero_word"], w),\n            "mean_tokens_forward_to_boundary": c["fwd_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n            "mean_tokens_back_to_boundary": c["back_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n        })\n    return S\n\n\ndef print_summary(S: dict) -> None:\n    print(f"\\n=== {S[\'tokenizer\']} ===")\n    print(f"sentences={S[\'sentences\']}  tokens={S[\'tokens\']}  "\n          f"tokens/word={S[\'tokens_per_word\']:.2f}  tokens/akshara={S[\'tokens_per_akshara\']:.2f}  "\n          f"tokens/char={S[\'tokens_per_char\']:.2f}")\n    print(f"tokens that START mid-akshara: {S[\'pct_tokens_start_mid_akshara\']:.1f}%   "\n          f"mid-UTF-8: {S[\'pct_tokens_start_mid_utf8\']:.1f}%")\n    print(f"all token cuts -> word boundary {S[\'pct_cuts_word_boundary\']:.1f}% | mid-word {S[\'pct_cuts_mid_word\']:.1f}% | "\n          f"mid-akshara {S[\'pct_cuts_mid_akshara\']:.1f}% | mid-UTF-8 {S[\'pct_cuts_mid_utf8\']:.1f}%")\n    if S["skipped_invalid_utf8"]:\n        print(f"WARNING: skipped {S[\'skipped_invalid_utf8\']} sentences (token bytes not valid UTF-8)")\n    hdr = ("k", "@word", "mid-word", "mid-aks", "mid-utf8", "inside-any", "words/draft", "0-word%", "fwd", "back")\n    print("\\n" + " ".join(f"{h:>11}" for h in hdr))\n    for r in S["per_k"]:\n        print(" ".join(f"{v:>11}" for v in (\n            r["k"], f"{r[\'pct_end_at_word_boundary\']:.1f}", f"{r[\'pct_end_mid_word\']:.1f}",\n            f"{r[\'pct_end_mid_akshara\']:.1f}", f"{r[\'pct_end_mid_utf8\']:.1f}",\n            f"{r[\'pct_end_anywhere_inside_word\']:.1f}", f"{r[\'mean_words_completed\']:.2f}",\n            f"{r[\'pct_drafts_completing_zero_words\']:.1f}",\n            f"{r[\'mean_tokens_forward_to_boundary\']:.2f}", f"{r[\'mean_tokens_back_to_boundary\']:.2f}")))\n    print("(columns 2-6 are % of k-token drafts; fwd/back = avg tokens to move to the nearest word boundary "\n          "when the draft ends inside a word)")\n\n\n# --------------------------------------------------------------------------- #\n# examples for slides\n# --------------------------------------------------------------------------- #\n\n\ndef _show(b: bytes) -> str:\n    try:\n        return b.decode("utf-8").replace(" ", "␣")  # visible space\n    except UnicodeDecodeError:\n        return "<" + b.hex() + ">"\n\n\ndef render_example(tb: Sequence[bytes], k: int) -> str:\n    """Render a tokenised sentence with fixed-k draft ends tagged by cut category."""\n    f = sentence_features(tb)\n    if f is None:\n        return "(invalid utf-8)"\n    out = []\n    for i, b in enumerate(tb):\n        out.append(_show(b))\n        if i < len(tb) - 1:\n            out.append(f" ‖[{TAG[f[\'cut\'][i]]}] " if (i + 1) % k == 0 else " | ")\n    return "".join(out)\n\n\nEXAMPLE_LEGEND = ("Legend: | token boundary,  ‖ end of a fixed-k draft,  tag = what the draft ends on: "\n                  "W=word boundary, w=inside word, a=inside akshara, u=inside UTF-8 char, ␣ = space\\n")\n\n# --------------------------------------------------------------------------- #\n# data\n# --------------------------------------------------------------------------- #\n\nDEMO = [\n    "भारत एक विशाल और विविधताओं से भरा देश है।",\n    "विद्यार्थियों ने विद्यालय में सूर्योदय के समय प्रार्थना की।",\n    "मौसम विभाग ने कल भारी बारिश की चेतावनी जारी की है।",\n    "वैज्ञानिकों का मानना है कि जलवायु परिवर्तन का प्रभाव कृषि पर पड़ेगा।",\n    "उसने अपने दोस्तों को खाने पर बुलाया और सबने मिलकर खाना खाया।",\n]\n\n\ndef load_sentences(path: str, field: str) -> list[str]:\n    out = []\n    with open(path, encoding="utf-8") as fh:\n        if path.endswith(".jsonl"):\n            for line in fh:\n                if line.strip():\n                    out.append(json.loads(line)[field])\n        else:\n            out = [ln.rstrip("\\n") for ln in fh]\n    return [unicodedata.normalize("NFC", s.strip()) for s in out if s.strip()]\n\n\ndef load_flores_hindi(split: str = "devtest") -> list[str]:\n    """FLORES+ hin_Deva sentences (NFC, stripped), as used in Test 1."""\n    from datasets import load_dataset\n    ds = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    return [unicodedata.normalize("NFC", s.strip()) for s in ds["text"] if s.strip()]\n',
    'vocab': '"""Boolean masks over the full vocabulary, built from token *bytes* (``boundaries.token_to_bytes``).\n\nMasks (all shape ``(len(tok),)``, dtype bool):\n\n- ``starts_space``: first byte in ``b" \\\\n\\\\t"``.\n- ``starts_punct``: first decoded char has Unicode category ``P*`` (``।``, ``,``, ``.``, ``"``, ...).\n- ``starts_mark``: first decoded char has category ``M*`` (matra, virama, nukta, anusvara,\n  candrabindu, visarga, ...) or is ZWJ / ZWNJ.\n- ``starts_cont``: first byte is a UTF-8 continuation byte (0x80-0xBF): the token starts mid-character.\n  Such a token is never ``starts_mark`` / ``starts_punct``.\n- ``is_special``: special / control tokens (all registered special tokens and eos ids).\n\nA token whose bytes start with a lead byte but do not contain a complete first character\n(e.g. ``b"\\\\xe0\\\\xa4"``) has an unknown first char and is in none of space/punct/mark/cont.\nSpecial tokens are only ``is_special``.\n"""\nfrom __future__ import annotations\n\nimport unicodedata\nfrom dataclasses import dataclass, field\nfrom typing import Iterable\n\nimport numpy as np\n\nfrom .boundaries import token_to_bytes\n\nZW = "\u200c\u200d"\nMASK_NAMES = ("starts_space", "starts_punct", "starts_mark", "starts_cont", "is_special")\n\n\ndef first_char(b: bytes) -> str:\n    """First complete character of ``b`` (\'\' if none). Continuation-byte starts give \'\'."""\n    if not b or 0x80 <= b[0] <= 0xBF:\n        return ""\n    return b[:4].decode("utf-8", errors="ignore")[:1]\n\n\ndef classify_token_bytes(b: bytes) -> dict[str, bool]:\n    """Mask flags for one non-special token given its raw bytes."""\n    out = dict(starts_space=False, starts_punct=False, starts_mark=False, starts_cont=False,\n               is_special=False)\n    if not b:\n        return out\n    if b[0] in b" \\n\\t":\n        out["starts_space"] = True\n        return out\n    if 0x80 <= b[0] <= 0xBF:\n        out["starts_cont"] = True\n        return out\n    ch = first_char(b)\n    if not ch:\n        return out\n    cat = unicodedata.category(ch)\n    if cat.startswith("P"):\n        out["starts_punct"] = True\n    elif cat.startswith("M") or ch in ZW:\n        out["starts_mark"] = True\n    return out\n\n\n@dataclass\nclass VocabMasks:\n    starts_space: np.ndarray\n    starts_punct: np.ndarray\n    starts_mark: np.ndarray\n    starts_cont: np.ndarray\n    is_special: np.ndarray\n    eos_ids: list[int] = field(default_factory=list)\n\n    @property\n    def size(self) -> int:\n        return len(self.starts_space)\n\n    @property\n    def is_eos(self) -> np.ndarray:\n        m = np.zeros(self.size, dtype=bool)\n        m[[i for i in self.eos_ids if i < self.size]] = True\n        return m\n\n    @property\n    def starts_newword(self) -> np.ndarray:\n        """Tokens that end the current word if they come next: space, punctuation or eos."""\n        return self.starts_space | self.starts_punct | self.is_eos\n\n    def padded(self, name: str, size: int) -> np.ndarray:\n        """Mask padded with False to ``size`` (model logits can be wider than ``len(tok)``)."""\n        m = getattr(self, name)\n        if size < len(m):\n            raise ValueError("size smaller than vocabulary")\n        return np.concatenate([m, np.zeros(size - len(m), dtype=bool)])\n\n    def as_dict(self) -> dict[str, np.ndarray]:\n        return {k: getattr(self, k) for k in MASK_NAMES}\n\n\ndef special_ids(tok) -> set[int]:\n    """All special/control token ids of a Hugging Face tokenizer."""\n    ids = set(getattr(tok, "all_special_ids", []) or [])\n    for i, at in (getattr(tok, "added_tokens_decoder", {}) or {}).items():\n        if getattr(at, "special", False):\n            ids.add(int(i))\n    return ids\n\n\ndef build_masks(tok, eos_ids: Iterable[int] = ()) -> VocabMasks:\n    """Build all masks for a Hugging Face tokenizer (size ``len(tok)``)."""\n    n = len(tok)\n    eos_ids = sorted(set(int(i) for i in eos_ids))\n    spec = special_ids(tok) | set(eos_ids)\n    toks = tok.convert_ids_to_tokens(list(range(n)))\n    arrs = {k: np.zeros(n, dtype=bool) for k in MASK_NAMES}\n    for i, t in enumerate(toks):\n        if i in spec or t is None:\n            arrs["is_special"][i] = True\n            continue\n        for k, v in classify_token_bytes(token_to_bytes(t)).items():\n            if v:\n                arrs[k][i] = True\n    return VocabMasks(**arrs, eos_ids=eos_ids)\n\n\ndef token_bytes_list(tok, ids) -> list[bytes]:\n    """Raw bytes of each id (special tokens give their literal text bytes)."""\n    return [token_to_bytes(t) for t in tok.convert_ids_to_tokens([int(i) for i in ids])]\n',
    'traces': '"""Trace schema (one target sentence), compressed ``.npz`` save/load, gold-label attachment.\n\nIndex convention (PLAN §3): index ``t`` refers to target token ``y[t]``; every draft feature at\n``t`` is a function of ``q_t``, the draft distribution over ``y[t]`` computed teacher-forced on\n``prompt + y[:t]``.\n\nStorage: per-position arrays of all sentences are concatenated into flat arrays and stored with\nan ``offsets`` array (sentence i occupies ``offsets[i]:offsets[i+1]``). ``meta`` is stored as a\nJSON string.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field, fields\nfrom pathlib import Path\n\nimport numpy as np\n\n# per-position 1-D arrays saved to disk (in this order)\nSCALAR_FIELDS = ("target_ids", "draft_argmax", "H", "top1", "top2", "p_target",\n                 "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                 "target_logp_of_draft_argmax")\n# per-position (n, K) arrays\nTOPK_FIELDS = ("draft_topk_ids", "draft_topk_logp", "target_topk_ids", "target_topk_logp")\nINT_FIELDS = {"target_ids", "draft_argmax", "draft_topk_ids", "target_topk_ids"}\nTOPK = 20\n\n\n@dataclass\nclass Trace:\n    """One sentence. All arrays have first dimension ``n``."""\n    id: int\n    n: int\n    target_ids: np.ndarray\n    draft_argmax: np.ndarray\n    H: np.ndarray            # draft entropy (nats)\n    top1: np.ndarray         # draft prob of its argmax\n    top2: np.ndarray         # draft prob of its 2nd-best token\n    p_target: np.ndarray     # q_t(y[t])\n    p_space: np.ndarray\n    p_punct: np.ndarray\n    p_eos: np.ndarray\n    p_newword: np.ndarray    # p_space + p_punct + p_eos\n    p_mark: np.ndarray\n    p_cont: np.ndarray\n    draft_topk_ids: np.ndarray\n    draft_topk_logp: np.ndarray\n    target_topk_ids: np.ndarray\n    target_topk_logp: np.ndarray\n    target_logp_of_draft_argmax: np.ndarray\n    match: np.ndarray = None  # derived: draft_argmax == target_ids\n    labels: dict = field(default_factory=dict, repr=False)  # gold labels (analysis only, not saved)\n\n    def __post_init__(self):\n        self.match = np.asarray(self.draft_argmax) == np.asarray(self.target_ids)\n        for f in SCALAR_FIELDS + TOPK_FIELDS:\n            a = getattr(self, f)\n            if len(a) != self.n:\n                raise ValueError(f"trace {self.id}: {f} has length {len(a)} != n={self.n}")\n\n\ndef save_traces(path, traces: list[Trace], meta: dict | None = None) -> None:\n    """Save traces as one compressed ``.npz`` (flat arrays + offsets)."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    lens = np.array([t.n for t in traces], dtype=np.int64)\n    offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    out = dict(ids=np.array([t.id for t in traces], dtype=np.int64), offsets=offsets,\n               meta=np.array(json.dumps(meta or {}, ensure_ascii=False)))\n    for f in SCALAR_FIELDS + TOPK_FIELDS:\n        dt = np.int32 if f in INT_FIELDS else np.float32\n        parts = [np.asarray(getattr(t, f), dtype=dt) for t in traces]\n        if parts:\n            out[f] = np.concatenate(parts, axis=0)\n        else:\n            out[f] = np.zeros((0, TOPK) if f in TOPK_FIELDS else (0,), dtype=dt)\n    np.savez_compressed(path, **out)\n\n\ndef load_traces(path) -> tuple[list[Trace], dict]:\n    """Inverse of ``save_traces``."""\n    z = np.load(path, allow_pickle=False)\n    meta = json.loads(str(z["meta"]))\n    offsets, ids = z["offsets"], z["ids"]\n    flat = {f: z[f] for f in SCALAR_FIELDS + TOPK_FIELDS}\n    traces = []\n    for i, sid in enumerate(ids):\n        a, b = int(offsets[i]), int(offsets[i + 1])\n        traces.append(Trace(id=int(sid), n=b - a, **{f: flat[f][a:b] for f in flat}))\n    return traces, meta\n\n\ndef summary(traces: list[Trace]) -> dict:\n    """Positions, agreement rate, mean draft entropy when match / mismatch."""\n    m = np.concatenate([t.match for t in traces])\n    H = np.concatenate([t.H for t in traces])\n    return dict(sentences=len(traces), positions=int(len(m)), agreement=float(m.mean()),\n                mean_H=float(H.mean()), mean_H_match=float(H[m].mean()) if m.any() else float("nan"),\n                mean_H_mismatch=float(H[~m].mean()) if (~m).any() else float("nan"))\n\n\ndef attach_word_labels(traces: list[Trace], tok) -> None:\n    """Attach gold word/akshara labels from the *target* tokens (fixes B5). Analysis only."""\n    from .boundaries import token_labels\n    from .vocab import special_ids, token_bytes_list\n    from .boundaries import _valid_utf8_prefix_len\n    spec = special_ids(tok)\n    for t in traces:\n        tb = token_bytes_list(tok, t.target_ids)\n        sp = [int(i) in spec for i in t.target_ids]\n        t.labels.update(token_labels(tb, special=sp))\n        full = b"".join(b for b, s in zip(tb, sp) if not s)\n        t.labels["text"] = full[:_valid_utf8_prefix_len(full)].decode("utf-8")  # text the char spans refer to\n\n\ndef trace_field_names() -> list[str]:\n    return [f.name for f in fields(Trace)]\n',
    'gpu_traces': '"""Target generation, teacher-forced draft/target passes and timing (Kaggle notebook 01).\n\nPorted from the legacy Part A (``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` cells 4-11).\nEverything here also runs on CPU (``scripts/smoke_cpu.py``) with a small model in fp32.\n\nOutputs (in ``out_dir``) for one split:\n  target_outputs_{split}.jsonl   id, src, ref, prompt_ids, out_ids, out_text   (append + resume)\n  _draft_tf_{split}.pkl          partial draft teacher-forced features         (resume)\n  _target_tf_{split}.pkl         partial target teacher-forced features        (resume)\n  traces_{split}.npz             final traces (``traces.save_traces``)\n  timing.json                    t_draft, t_target, c, verify(g)               (skipped if present)\n  run_info_{split}.json\n"""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport os\nimport pickle\nimport platform\nimport shutil\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .traces import TOPK, Trace, save_traces, summary\nfrom .vocab import VocabMasks, build_masks\n\nPROMPT_TEMPLATE = "Translate the following English sentence into Hindi. Output only the Hindi translation.\\n\\n{s}"\n\n\n# --------------------------------------------------------------------------- #\n# loading\n# --------------------------------------------------------------------------- #\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef sync_all() -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\ndef gpu_report(tag: str = "") -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        print(tag, " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB allocated"\n                              for i in range(torch.cuda.device_count())))\n\n\ndef from_pretrained(name: str, dtype, **kw):\n    """``AutoModelForCausalLM.from_pretrained`` across transformers versions (dtype vs torch_dtype)."""\n    from transformers import AutoModelForCausalLM\n    try:\n        return AutoModelForCausalLM.from_pretrained(name, dtype=dtype, **kw).eval()\n    except TypeError:\n        return AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, **kw).eval()\n\n\ndef load_tokenizer(name: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(name)\n    tok.padding_side = "left"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_target(name: str, dtype, max_memory: dict | None = None, device_map="auto"):\n    """Target model, layer-split over the available GPUs (``device_map=\'auto\'``) or on CPU."""\n    torch = _torch()\n    if torch.cuda.is_available():\n        m = from_pretrained(name, dtype, device_map=device_map, max_memory=max_memory,\n                            low_cpu_mem_usage=True)\n        print("modules per device:", dict(Counter(m.hf_device_map.values())))\n    else:\n        m = from_pretrained(name, dtype)\n    return m\n\n\ndef load_draft(name: str, dtype, device: str):\n    return from_pretrained(name, dtype).to(device)\n\n\ndef get_eos_ids(model, tok) -> list[int]:\n    """Stop ids: generation-config eos plus ``<|eot_id|>`` (Llama-3 instruct end of turn)."""\n    eos = model.generation_config.eos_token_id\n    ids = set(eos if isinstance(eos, list) else [eos])\n    eot = tok.convert_tokens_to_ids("<|eot_id|>")\n    if eot is not None and eot != tok.unk_token_id:\n        ids.add(eot)\n    return sorted(int(i) for i in ids if i is not None)\n\n\ndef check_draft_target_compat(draft, target, tok_d, tok_t, probe: str) -> None:\n    """Draft and target must share a vocabulary (legacy asserts, kept)."""\n    assert draft.config.vocab_size == target.config.vocab_size, "vocab sizes differ"\n    probe = probe + " नमस्ते दुनिया. Hello world!"\n    assert (tok_d(probe, add_special_tokens=False).input_ids\n            == tok_t(probe, add_special_tokens=False).input_ids), "tokenizers disagree"\n    print("draft/target tokenizers match")\n\n\n# --------------------------------------------------------------------------- #\n# data + prompts\n# --------------------------------------------------------------------------- #\n\ndef load_flores_pairs(split: str, n: int | None = None):\n    """FLORES+ eng_Latn -> hin_Deva rows ``(id, src, ref)`` for ``split`` in {dev, devtest}."""\n    from datasets import load_dataset\n    assert split in ("dev", "devtest"), split\n    en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split=split)\n    hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    hi_by_id = {r["id"]: r["text"] for r in hi}\n    rows = [(int(r["id"]), r["text"], hi_by_id[r["id"]]) for r in en]\n    return rows[:n] if n else rows\n\n\ndef build_prompt(tok, s: str) -> str:\n    msgs = [{"role": "user", "content": PROMPT_TEMPLATE.format(s=s)}]\n    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef read_jsonl(path) -> list[dict]:\n    path = Path(path)\n    if not path.exists():\n        return []\n    with open(path, encoding="utf-8") as fh:\n        return [json.loads(l) for l in fh if l.strip()]\n\n\n# --------------------------------------------------------------------------- #\n# 1. batched greedy target generation (resumable)\n# --------------------------------------------------------------------------- #\n\ndef generate_targets(target, tok, rows, out_jsonl, batch: int, max_new: int, eos_ids,\n                     device: str, progress: bool = True) -> list[dict]:\n    """Greedy, batched (left padding) target outputs, appended to ``out_jsonl`` after every batch.\n\n    Resumes if ``out_jsonl`` holds a prefix of ``rows`` (by id); otherwise starts over.\n    """\n    torch = _torch()\n    from tqdm.auto import tqdm\n    out_jsonl = Path(out_jsonl)\n    ids = [r[0] for r in rows]\n    prompts = [build_prompt(tok, r[1]) for r in rows]\n    records = read_jsonl(out_jsonl)\n    if [r["id"] for r in records] != ids[:len(records)]:\n        print("existing output does not match this run; starting over")\n        records = []\n    print(f"resuming: {len(records)} / {len(rows)} sentences already done")\n    eos_set = set(eos_ids)\n    with open(out_jsonl, "a" if records else "w", encoding="utf-8") as f:\n        rng = range(len(records), len(prompts), batch)\n        for i in (tqdm(rng, desc="generate") if progress else rng):\n            enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True,\n                      add_special_tokens=False).to(device)\n            with torch.no_grad():\n                out = target.generate(**enc, do_sample=False, temperature=None, top_p=None,\n                                      max_new_tokens=max_new, eos_token_id=list(eos_set),\n                                      pad_token_id=tok.pad_token_id)\n            gen = out[:, enc.input_ids.shape[1]:].tolist()\n            for j, g in enumerate(gen):\n                cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))  # keep eos\n                g = g[:cut]\n                pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()      # unpadded prompt\n                rid, src, ref = rows[i + j]\n                rec = dict(id=rid, src=src, ref=ref, prompt_ids=pid, out_ids=g,\n                           out_text=tok.decode(g, skip_special_tokens=True).strip())\n                records.append(rec)\n                f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n    n_hit = sum(len(r["out_ids"]) >= max_new for r in records)\n    print(f"{len(records)} target outputs in {out_jsonl.name}; {n_hit} hit MAX_NEW={max_new}")\n    return records\n\n\ndef generate_unbatched(model, tok, prompt_ids: list[int], max_new: int, eos_ids, device: str) -> list[int]:\n    """Plain greedy generation for one prompt, no padding (reference for losslessness, B9)."""\n    torch = _torch()\n    x = torch.tensor([prompt_ids], device=device)\n    with torch.no_grad():\n        out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), do_sample=False,\n                             temperature=None, top_p=None, max_new_tokens=max_new,\n                             eos_token_id=list(eos_ids), pad_token_id=tok.pad_token_id)\n    g = out[0, len(prompt_ids):].tolist()\n    eos_set = set(eos_ids)\n    cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))\n    return g[:cut]\n\n\n# --------------------------------------------------------------------------- #\n# 2./3. teacher-forced passes\n# --------------------------------------------------------------------------- #\n\ndef _mask_matrix(masks: VocabMasks, vocab_size: int, device):\n    """(V, 5) float32 matrix: columns space, punct, eos, mark, cont."""\n    torch = _torch()\n    cols = [masks.padded("starts_space", vocab_size), masks.padded("starts_punct", vocab_size),\n            np.concatenate([masks.is_eos, np.zeros(vocab_size - masks.size, bool)]),\n            masks.padded("starts_mark", vocab_size), masks.padded("starts_cont", vocab_size)]\n    return torch.tensor(np.stack(cols, 1).astype(np.float32), device=device)\n\n\ndef _logits_for_output(model, prompt_ids, out_ids, device):\n    """Logits (float32) predicting each output token: row t is q_t (conditioned on prompt + y[:t])."""\n    torch = _torch()\n    x = torch.tensor([list(prompt_ids) + list(out_ids)], device=device)\n    with torch.no_grad():\n        lg = model(x).logits[0, len(prompt_ids) - 1:-1]\n    return lg.float()\n\n\ndef draft_features(draft, rec: dict, M, device: str, topk: int = TOPK) -> dict[str, np.ndarray]:\n    """Per-position draft features for one record (all float32 on host)."""\n    torch = _torch()\n    lg = _logits_for_output(draft, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    p = lp.exp()\n    H = -(p * lp).sum(-1)\n    t2 = p.topk(2, -1).values\n    tk = lp.topk(topk, -1)\n    y = torch.tensor(rec["out_ids"], device=lg.device)\n    P = p @ M.to(lg.device)                       # (n, 5): space, punct, eos, mark, cont\n    f = dict(draft_argmax=lp.argmax(-1), H=H, top1=t2[:, 0], top2=t2[:, 1],\n             p_target=p.gather(1, y[:, None]).squeeze(1),\n             p_space=P[:, 0], p_punct=P[:, 1], p_eos=P[:, 2], p_newword=P[:, 0] + P[:, 1] + P[:, 2],\n             p_mark=P[:, 3], p_cont=P[:, 4], draft_topk_ids=tk.indices, draft_topk_logp=tk.values)\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef target_features(target, rec: dict, draft_argmax: np.ndarray, device: str,\n                    topk: int = TOPK) -> dict[str, np.ndarray]:\n    torch = _torch()\n    lg = _logits_for_output(target, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    tk = lp.topk(topk, -1)\n    da = torch.tensor(np.asarray(draft_argmax), device=lg.device, dtype=torch.long)\n    f = dict(target_topk_ids=tk.indices, target_topk_logp=tk.values,\n             target_logp_of_draft_argmax=lp.gather(1, da[:, None]).squeeze(1))\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef _resumable_pass(records, fn, partial_path, desc: str, save_every: int = 50, progress=True):\n    """Run ``fn(rec)`` for every record, pickling partial results so a dead session can resume."""\n    from tqdm.auto import tqdm\n    partial_path = Path(partial_path)\n    done: dict = {}\n    if partial_path.exists():\n        with open(partial_path, "rb") as fh:\n            done = pickle.load(fh)\n        print(f"{desc}: resuming with {len(done)} / {len(records)} done")\n    it = tqdm(records, desc=desc) if progress else records\n    since = 0\n    for rec in it:\n        if rec["id"] in done:\n            continue\n        done[rec["id"]] = fn(rec)\n        since += 1\n        if since >= save_every:\n            _atomic_pickle(done, partial_path)\n            since = 0\n    _atomic_pickle(done, partial_path)\n    return done\n\n\ndef _atomic_pickle(obj, path: Path) -> None:\n    tmp = path.with_suffix(".tmp")\n    with open(tmp, "wb") as fh:\n        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)\n    os.replace(tmp, path)\n\n\ndef run_draft_pass(draft, records, masks: VocabMasks, out_dir, split: str, device: str, progress=True):\n    vocab = draft.get_output_embeddings().weight.shape[0]\n    M = _mask_matrix(masks, vocab, device)\n    return _resumable_pass(records, lambda r: draft_features(draft, r, M, device),\n                           Path(out_dir) / f"_draft_tf_{split}.pkl", "draft teacher-forced", progress=progress)\n\n\ndef run_target_pass(target, records, draft_feats: dict, out_dir, split: str, device: str, progress=True):\n    return _resumable_pass(records, lambda r: target_features(target, r, draft_feats[r["id"]]["draft_argmax"], device),\n                           Path(out_dir) / f"_target_tf_{split}.pkl", "target teacher-forced", progress=progress)\n\n\ndef assemble_traces(records, draft_feats: dict, target_feats: dict) -> list[Trace]:\n    out = []\n    for r in records:\n        d, t = draft_feats[r["id"]], target_feats[r["id"]]\n        out.append(Trace(id=int(r["id"]), n=len(r["out_ids"]), target_ids=np.asarray(r["out_ids"], np.int32),\n                         **d, **t))\n    return out\n\n\ndef target_argmax_check(traces: list[Trace]) -> dict:\n    """Rate at which the unpadded teacher-forced target argmax equals the generated y[t]."""\n    agree, diffs, total = 0, [], 0\n    for t in traces:\n        am = t.target_topk_ids[:, 0]\n        eq = am == t.target_ids\n        agree += int(eq.sum())\n        total += t.n\n        for pos in np.flatnonzero(~eq):\n            lp = t.target_topk_logp[pos]\n            diffs.append(dict(id=t.id, pos=int(pos), y=int(t.target_ids[pos]), argmax=int(am[pos]),\n                              margin=float(lp[0] - lp[1])))\n    return dict(rate=agree / max(total, 1), positions=total, n_diff=len(diffs), diffs=diffs)\n\n\n# --------------------------------------------------------------------------- #\n# 4. timing\n# --------------------------------------------------------------------------- #\n\ndef time_fwd(model, q: int = 1, ctx: int = 150, steps: int = 60, reps: int = 3, device: str = "cuda:0") -> float:\n    """Median latency (s) of a forward with ``q`` new tokens on top of a KV cache of ``ctx`` tokens."""\n    torch = _torch()\n    ts = []\n    with torch.no_grad():\n        for _ in range(reps):\n            x = torch.randint(1000, 20000, (1, ctx), device=device)\n            o = model(x, use_cache=True)\n            pkv = o.past_key_values\n            for s in range(steps + 10):\n                inp = torch.randint(1000, 20000, (1, q), device=device)\n                sync_all()\n                t = time.perf_counter()\n                o = model(inp, past_key_values=pkv, use_cache=True)\n                sync_all()\n                dt = time.perf_counter() - t\n                pkv = o.past_key_values\n                if s >= 10:\n                    ts.append(dt)\n    return float(np.median(ts))\n\n\ndef run_timing(draft, target, out_path, device: str, setup: dict, steps: int = 60, reps: int = 3,\n               max_g: int = 8) -> dict:\n    """Legacy timing (c and verify(g), g=1..max_g) plus draft q=1 at ctx 150 and 400."""\n    out_path = Path(out_path)\n    if out_path.exists():\n        print(f"{out_path.name} exists, timing skipped")\n        return json.loads(out_path.read_text())\n    t_d = time_fwd(draft, 1, steps=steps, reps=reps, device=device)\n    t_t = time_fwd(target, 1, steps=steps, reps=reps, device=device)\n    print(f"draft decode step: {t_d * 1000:.2f} ms | target decode step: {t_t * 1000:.2f} ms | c = {t_d / t_t:.3f}")\n    ver = {g: time_fwd(target, g + 1, steps=steps, reps=reps, device=device) for g in range(1, max_g + 1)}\n    print({g: f"{v * 1000:.1f} ms" for g, v in ver.items()})\n    extra = {str(ctx): time_fwd(draft, 1, ctx=ctx, steps=steps, reps=reps, device=device) for ctx in (150, 400)}\n    T = dict(t_draft=t_d, t_target=t_t, c=t_d / t_t, t_target_verify={str(g): v for g, v in ver.items()},\n             t_draft_by_ctx=extra, setup=setup)\n    out_path.write_text(json.dumps(T, indent=1))\n    return T\n\n\n# --------------------------------------------------------------------------- #\n# run info + zip\n# --------------------------------------------------------------------------- #\n\ndef run_info(git_hash: str, **extra) -> dict:\n    torch = _torch()\n    import transformers\n    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []\n    return dict(git_hash=git_hash, torch=torch.__version__, transformers=transformers.__version__,\n                python=platform.python_version(), gpus=gpus, time=time.strftime("%Y-%m-%d %H:%M:%S"), **extra)\n\n\ndef zip_dir(src_dir, zip_path, include=None) -> str:\n    """Zip files of ``src_dir`` (only names in ``include`` if given; flat, no folder) to ``zip_path``.\n    Never deletes or moves anything in ``src_dir``. Returns the zip path."""\n    import zipfile\n    src_dir, zip_path = Path(src_dir), Path(zip_path)\n    names = list(include) if include is not None else sorted(p.name for p in src_dir.iterdir() if p.is_file())\n    missing = [n for n in names if not (src_dir / n).exists()]\n    if missing:\n        raise FileNotFoundError(f"cannot zip, missing in {src_dir}: {missing}")\n    tmp = zip_path.with_name(zip_path.name + ".part")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for n in names:\n            z.write(src_dir / n, arcname=n)\n    os.replace(tmp, zip_path)\n    return str(zip_path)\n\n\ndef free(*objs) -> None:\n    for o in objs:\n        del o\n    gc.collect()\n    torch = _torch()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\n# --------------------------------------------------------------------------- #\n# full pipeline for one split (used by notebook 01 and the smoke test)\n# --------------------------------------------------------------------------- #\n\ndef run_split(*, split: str, out_dir, target, draft, tok, tok_d, n: int | None, batch: int,\n              max_new: int, device: str, git_hash: str, timing_kw: dict | None = None,\n              setup_extra: dict | None = None, rows=None, progress: bool = True,\n              zip_name: str | None = None) -> dict:\n    """Steps 1-6 of PLAN T2 for one split. Returns a dict with paths and summaries.\n\n    timing_kw: passed to ``run_timing`` (steps, reps, max_g); setup_extra: added to timing.json setup\n    (e.g. max_memory, GPU names).\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = rows if rows is not None else load_flores_pairs(split, n)\n    eos_ids = get_eos_ids(target, tok)\n    print(f"{split}: {len(rows)} sentences; eos ids {eos_ids}")\n\n    records = generate_targets(target, tok, rows, out_dir / f"target_outputs_{split}.jsonl",\n                               batch, max_new, eos_ids, device, progress=progress)\n    check_draft_target_compat(draft, target, tok_d, tok, records[0]["out_text"])\n\n    masks = build_masks(tok, eos_ids)\n    dfe = run_draft_pass(draft, records, masks, out_dir, split, device, progress=progress)\n    tfe = run_target_pass(target, records, dfe, out_dir, split, device, progress=progress)\n    traces = assemble_traces(records, dfe, tfe)\n    chk = target_argmax_check(traces)\n    print(f"target teacher-forced argmax == y[t]: {chk[\'rate\']:.4f} ({chk[\'n_diff\']} positions differ)")\n    for d in chk["diffs"][:20]:\n        print("   ", d)\n    if chk["rate"] < 0.995:\n        print("WARNING: target argmax agreement below 0.995")\n\n    import transformers\n    meta = dict(split=split, target=getattr(target.config, "_name_or_path", ""),\n                draft=getattr(draft.config, "_name_or_path", ""), max_new=max_new, batch=batch,\n                eos_ids=eos_ids, git_hash=git_hash, transformers=transformers.__version__,\n                target_argmax_check={k: v for k, v in chk.items() if k != "diffs"},\n                target_argmax_diffs=chk["diffs"][:500])\n    npz = out_dir / f"traces_{split}.npz"\n    save_traces(npz, traces, meta)\n    S = summary(traces)\n    print(f"positions: {S[\'positions\']} | agreement: {S[\'agreement\']:.3f} | mean H: {S[\'mean_H\']:.3f} "\n          f"(match {S[\'mean_H_match\']:.3f}, mismatch {S[\'mean_H_mismatch\']:.3f})")\n\n    setup = dict(target=meta["target"], draft=meta["draft"], dtype=str(next(target.parameters()).dtype),\n                 device=device, **(setup_extra or {}))\n    T = run_timing(draft, target, out_dir / "timing.json", device, setup, **(timing_kw or {}))\n\n    info = run_info(git_hash, split=split, n_sentences=len(rows), summary=S,\n                    target=meta["target"], draft=meta["draft"])\n    (out_dir / f"run_info_{split}.json").write_text(json.dumps(info, indent=1, ensure_ascii=False))\n    zip_name = zip_name or f"traces_v2_{split}"\n    zp = zip_dir(out_dir, out_dir.parent / f"{zip_name}.zip",\n                 include=[npz.name, f"target_outputs_{split}.jsonl", "timing.json", f"run_info_{split}.json"])\n    return dict(zip=zp, npz=str(npz), summary=S, timing=T, target_check=chk)\n\n\n# --------------------------------------------------------------------------- #\n# notebook 01 entry point (also called by scripts/smoke_cpu.py)\n# --------------------------------------------------------------------------- #\n\nDEFAULT_CONFIG_01 = dict(\n    SPLIT="dev",                                    # "dev" or "devtest": one split per run\n    N=None,                                         # None = all sentences of the split\n    BATCH=16, MAX_NEW=256,\n    SMOKE=False,                                    # True: CPU, fp32, one model as target+draft\n    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",\n    DRAFT="meta-llama/Llama-3.2-1B-Instruct",\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    OUT_ROOT="/kaggle/working",\n    GIT_HASH="unknown",\n    TIMING=dict(steps=60, reps=3, max_g=8),\n    RESUME_FROM=None,   # folder with partial outputs of an earlier (dead) run, e.g. /kaggle/input/<...>\n)\n\n\ndef _prepare_out_dir(out_dir: Path, out_root: Path, resume_from) -> None:\n    """Copy partial outputs from ``resume_from`` (searched recursively) and reuse a timing.json\n    measured by another split\'s run in this session."""\n    out_dir.mkdir(parents=True, exist_ok=True)\n    if resume_from:\n        src = Path(resume_from)\n        for f in src.rglob("*"):\n            if f.is_file() and f.parent.name == out_dir.name and not (out_dir / f.name).exists():\n                shutil.copy2(f, out_dir / f.name)\n                print("resume: copied", f)\n    if not (out_dir / "timing.json").exists():\n        for other in sorted(out_root.glob("traces_v2_*/timing.json")):\n            shutil.copy2(other, out_dir / "timing.json")\n            print("reusing timing from", other)\n            break\n\n\ndef notebook01_main(cfg: dict) -> dict:\n    """Run PLAN T2 steps 1-6 for ``cfg[\'SPLIT\']``. Writes ``{OUT_ROOT}/traces_v2_{split}/`` and\n    ``{OUT_ROOT}/traces_v2_{split}.zip``. With ``SMOKE=True`` the draft model is used as both\n    target and draft (fp32, CPU, loaded once) and only 3 sentences are processed."""\n    torch = _torch()\n    cfg = {**DEFAULT_CONFIG_01, **cfg}\n    split = cfg["SPLIT"]\n    out_dir = Path(cfg["OUT_ROOT"]) / f"traces_v2_{split}"\n    _prepare_out_dir(out_dir, Path(cfg["OUT_ROOT"]), cfg["RESUME_FROM"])\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        tok = load_tokenizer(cfg["DRAFT"])\n        draft = target = load_draft(cfg["DRAFT"], dtype, device)\n        tok_d = tok\n        n = cfg["N"] or 3\n        setup_extra = dict(smoke=True)\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        for i in range(torch.cuda.device_count()):\n            p = torch.cuda.get_device_properties(i)\n            print(f"GPU{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")\n        device, dtype = "cuda:0", torch.float16\n        tok = load_tokenizer(cfg["TARGET"])\n        tok_d = load_tokenizer(cfg["DRAFT"])\n        target = load_target(cfg["TARGET"], dtype, max_memory=cfg["MAX_MEMORY"])\n        gpu_report("after loading target:")\n        draft = load_draft(cfg["DRAFT"], dtype, device)\n        gpu_report("after loading draft:")\n        n = cfg["N"]\n        setup_extra = dict(max_memory={str(k): v for k, v in cfg["MAX_MEMORY"].items()},\n                           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\n    res = run_split(split=split, out_dir=out_dir, target=target, draft=draft, tok=tok, tok_d=tok_d,\n                    n=n, batch=cfg["BATCH"], max_new=cfg["MAX_NEW"], device=device,\n                    git_hash=cfg["GIT_HASH"], timing_kw=cfg["TIMING"], setup_extra=setup_extra)\n    print("\\n" + "=" * 70)\n    print(f"DONE. Download  {Path(res[\'zip\']).name}  from the notebook\'s Output tab")\n    print(f"and unzip it into  data/traces_v2/  in the repo (files land directly in that folder).")\n    print("=" * 70)\n    return res\n',
    'simulator': '"""Trace simulator for greedy speculative decoding (PLAN T4, fixes B3, B8).\n\nPorted from ``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` (cells 14-17), with:\n  1. the ``Policy`` interface ``stop(view, p, g)`` called with g = 1..cap-1 after each drafted token\n     (dynamic policies always draft >= 1 token; NoDraft / FixedK(0) / Oracle decide g = 0 via\n     ``initial_stop(view, p)``);\n  2. a causality guard (``TraceView``) - see PLAN §3;\n  3. lookahead cost: a lookahead-1 policy that stops by itself pays one extra draft forward ``c``;\n  4. per-sentence emitted / cost arrays and round records;\n  5. measured verification cost as the default.\n\nExactness for greedy decoding: within a round the draft\'s own tokens coincide with the target\'s up\nto the first mismatch, and everything after it is rejected, so accepted = leading matches.\nOnly the stopping decision after a mismatch can differ from a real loop (teacher-forcing\napproximation, PLAN §3).\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable, Sequence\n\nimport numpy as np\n\nfrom .traces import Trace\n\nGAMMA_MAX = 8\n\n# trace fields a policy may read through ``view.feat`` (functions of q_t only)\nDRAFT_FEATURES = {"H", "top1", "top2", "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                  "draft_argmax", "draft_topk_ids", "draft_topk_logp"}\n# never readable by non-oracle policies\nGOLD_FIELDS = {"match", "target_ids", "p_target", "target_topk_ids", "target_topk_logp",\n               "target_logp_of_draft_argmax"}\n\n\nclass CausalityError(RuntimeError):\n    """A policy read information a real decoder would not have at that moment."""\n\n\nclass TraceView:\n    """Read-only, causality-checked view of one trace for policies.\n\n    The simulator sets ``limit = p + g - 1 + policy.lookahead`` and ``p`` before each call.\n      feat(name, idx): draft feature at index idx (idx <= limit).\n      token(idx):      committed target token for idx < p, the draft\'s own token for p <= idx <= p+g-1\n                       (idx <= limit; for lookahead policies idx = p+g gives the draft argmax at p+g).\n      gold(name, idx): any field incl. match / target ids / labels; oracle policies only.\n    """\n\n    __slots__ = ("trace", "limit", "p", "is_oracle", "n", "_labels")\n\n    def __init__(self, trace: Trace, is_oracle: bool = False):\n        self.trace, self.is_oracle, self.n = trace, is_oracle, trace.n\n        self.limit = -1\n        self.p = 0\n        self._labels = trace.labels\n\n    def _check(self, idx: int) -> None:\n        if idx > self.limit and not self.is_oracle:\n            raise CausalityError(f"read index {idx} > limit {self.limit} (p={self.p})")\n        if idx < 0 or idx >= self.n:\n            raise IndexError(idx)\n\n    def feat(self, name: str, idx: int):\n        if name not in DRAFT_FEATURES and not self.is_oracle:\n            raise CausalityError(f"policy may not read \'{name}\' via feat()")\n        self._check(idx)\n        return getattr(self.trace, name)[idx]\n\n    def token(self, idx: int) -> int:\n        self._check(idx)\n        if idx < self.p:\n            return int(self.trace.target_ids[idx])\n        return int(self.trace.draft_argmax[idx])\n\n    def gold(self, name: str, idx: int):\n        if not self.is_oracle:\n            raise CausalityError(f"non-oracle policy read gold field \'{name}\'")\n        if name in self._labels:\n            return self._labels[name][idx]\n        return getattr(self.trace, name)[idx]\n\n\nclass Policy:\n    name: str = "policy"\n    lookahead: int = 0          # 0 or 1\n    is_oracle: bool = False\n\n    def initial_stop(self, view: TraceView, p: int) -> bool:\n        """Decision at g = 0 (before drafting; view.limit = p - 1). Only NoDraft / FixedK(0) and the\n        oracle can return True: every other dynamic policy drafts at least one token."""\n        return False\n\n    def stop(self, view: TraceView, p: int, g: int) -> bool:  # g >= 1\n        raise NotImplementedError\n\n    def reset(self, view: TraceView) -> None:\n        """Called once per sentence (policies with per-sentence caches)."""\n\n    def params(self) -> dict:\n        return {}\n\n    def __repr__(self):\n        ps = ", ".join(f"{k}={v}" for k, v in self.params().items())\n        return f"{self.name}({ps})"\n\n\n# --------------------------------------------------------------------------- #\n# cost model\n# --------------------------------------------------------------------------- #\n\ndef load_timing(path) -> dict:\n    return json.loads(Path(path).read_text())\n\n\nclass MeasuredVerify:\n    """Measured verification cost for g drafted tokens (g+1 target tokens) relative to one decode\n    step, linearly interpolated, clipped below at 1 (dips are timing noise). verify(0) = 1.\n    A class (not a closure) so it can be sent to worker processes."""\n\n    def __init__(self, timing: dict):\n        self.gs = sorted(int(g) for g in timing["t_target_verify"])\n        self.vs = [max(1.0, timing["t_target_verify"][str(g)] / timing["t_target"]) for g in self.gs]\n        self.table = {0: 1.0}\n        for g in range(1, max(self.gs) + 1):\n            self.table[g] = float(np.interp(g, self.gs, self.vs))\n\n    def __call__(self, g: int) -> float:\n        return self.table[g] if g in self.table else float(np.interp(g, self.gs, self.vs))\n\n\ndef make_verify(timing: dict) -> MeasuredVerify:\n    return MeasuredVerify(timing)\n\n\ndef unit_verify(g: int) -> float:\n    return 1.0\n\n\ndef formula_speedup(alpha: float, gamma: int, c: float) -> float:\n    """f = (1 - a^(g+1)) / ((1 - a)(g c + 1)), i.i.d. acceptance assumption (legacy)."""\n    exp_tokens = (gamma + 1) if alpha >= 1 else (1 - alpha ** (gamma + 1)) / (1 - alpha)\n    return exp_tokens / (gamma * c + 1)\n\n\n# --------------------------------------------------------------------------- #\n# simulator\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass SimResult:\n    rounds: int = 0\n    drafted: int = 0\n    accepted: int = 0\n    rejections: int = 0\n    emitted: int = 0\n    cost: float = 0.0\n    lookahead_charges: int = 0\n    policy_stops: int = 0\n    draft_lens: list = field(default_factory=list)\n    sent_ids: np.ndarray = None\n    sent_emitted: np.ndarray = None\n    sent_cost: np.ndarray = None\n    records: list = field(default_factory=list)   # (sentence_id, p, g, accepted, stopped_by_policy)\n\n\ndef simulate(traces: Sequence[Trace], policy: Policy, gamma_max: int = GAMMA_MAX, c: float = 0.334,\n             verify: Callable[[int], float] | None = None, charge_lookahead: bool = True,\n             record_rounds: bool = True) -> SimResult:\n    """Simulate greedy speculative decoding of every trace with ``policy``.\n\n    One round at committed position p: draft g tokens (cap = min(gamma_max, n - p)), accept the\n    leading matches a, emit min(a + 1, n - p) tokens, cost g*c + verify(g)\n    (+ c if the policy has lookahead 1, stopped by itself before the cap, and charge_lookahead).\n    """\n    verify = verify or unit_verify\n    R = SimResult()\n    ns = len(traces)\n    R.sent_ids = np.array([t.id for t in traces])\n    R.sent_emitted = np.zeros(ns, dtype=np.int64)\n    R.sent_cost = np.zeros(ns, dtype=np.float64)\n    la = int(policy.lookahead)\n    for si, tr in enumerate(traces):\n        n, match = tr.n, tr.match\n        view = TraceView(tr, policy.is_oracle)\n        policy.reset(view)\n        p = 0\n        s_emit, s_cost = 0, 0.0\n        while p < n:\n            r = n - p\n            cap = min(gamma_max, r)\n            view.p = p\n            g = 0\n            by_policy = False\n            view.limit = p - 1\n            if cap > 0 and not policy.initial_stop(view, p):\n                g = 1\n                while g < cap:\n                    view.limit = p + g - 1 + la\n                    if policy.stop(view, p, g):\n                        by_policy = True\n                        break\n                    g += 1\n            elif cap > 0:\n                by_policy = True\n            a = 0\n            while a < g and match[p + a]:\n                a += 1\n            gained = min(a + 1, r)\n            cost = g * c + verify(g)\n            if la and by_policy and g > 0 and charge_lookahead:\n                cost += c\n                R.lookahead_charges += 1\n            R.rounds += 1\n            R.drafted += g\n            R.accepted += a\n            R.rejections += int(a < g)\n            R.emitted += gained\n            R.cost += cost\n            R.policy_stops += int(by_policy)\n            R.draft_lens.append(g)\n            if record_rounds:\n                R.records.append((tr.id, p, g, a, by_policy))\n            s_emit += gained\n            s_cost += cost\n            p += gained\n        R.sent_emitted[si] = s_emit\n        R.sent_cost[si] = s_cost\n    return R\n\n\ndef summarize(res: SimResult) -> dict:\n    """Legacy summary metrics."""\n    return dict(\n        acceptance_rate=res.accepted / max(res.drafted, 1),                    # accepted / drafted\n        alpha_hat=res.accepted / max(res.accepted + res.rejections, 1),        # per-token accept prob (geometric MLE)\n        tokens_per_verif=res.emitted / max(res.rounds, 1),\n        mean_draft_len=res.drafted / max(res.rounds, 1),\n        speedup=res.emitted / res.cost if res.cost else float("nan"),\n    )\n\n\n# --------------------------------------------------------------------------- #\n# legacy protocol (sanity check against the PDF numbers, PLAN T4)\n# --------------------------------------------------------------------------- #\n\ndef legacy_protocol(traces: Sequence[Trace], c: float, seed: int = 0) -> dict:\n    """Reproduce the old evaluation: random half split of devtest (default_rng(0).permutation),\n    entropy grid lam = linspace(-1.5, 0.95, 40), beta in {0.5, 1, 2}, unit verify, test on the 2nd half.\n    Expected (PDF): fixed k=2 ~ 1.36, entropy ~ 1.45 (within ~0.03 after regeneration)."""\n    from .policies import Entropy, FixedK, Oracle\n    idx = np.random.default_rng(seed).permutation(len(traces))\n    dev = [traces[i] for i in idx[: len(idx) // 2]]\n    test = [traces[i] for i in idx[len(idx) // 2:]]\n    ks = range(2, 9)\n    best_k = max(ks, key=lambda k: summarize(simulate(dev, FixedK(k), k, c, record_rounds=False))["speedup"])\n    rows = []\n    for beta in (0.5, 1.0, 2.0):\n        for lam in np.round(np.linspace(-1.5, 0.95, 40), 3):\n            m = summarize(simulate(dev, Entropy(float(lam), beta), GAMMA_MAX, c, record_rounds=False))\n            rows.append((m["speedup"], float(lam), beta))\n    _, lam, beta = max(rows)\n    fixed_all = {k: summarize(simulate(traces, FixedK(k), k, c, record_rounds=False)) for k in ks}\n    return dict(\n        best_k_dev=best_k,\n        fixed_test=summarize(simulate(test, FixedK(best_k), best_k, c, record_rounds=False)),\n        fixed_k2_all=fixed_all[2],\n        entropy_params=dict(lam=lam, beta=beta),\n        entropy_test=summarize(simulate(test, Entropy(lam, beta), GAMMA_MAX, c, record_rounds=False)),\n        oracle_test=summarize(simulate(test, Oracle(), GAMMA_MAX, c, record_rounds=False)),\n        expected=dict(fixed_k2=1.361, entropy=1.452, oracle=1.905),\n    )\n',
    'policies': '"""Stopping policies (PLAN T5, T6). Each reads only through ``simulator.TraceView``.\n\nCausal (lookahead 0) policies read draft features at indices <= p+g-1. Lookahead-1 policies also read\nindex p+g (the next draft distribution q_{p+g}, one extra draft forward; the simulator charges it\nwhen the policy stops by itself). Oracles (``is_oracle``) may read gold fields; they are upper bounds.\n\n"Extend" semantics (AksharaSafeEntropy, Hybrid, MorphEntropy): let g0 be the first g at which the\nentropy rule fired in this round (recomputed from H[p..p+g-1], so there is no hidden state). At every\ng >= g0 the policy stops if its linguistic condition holds at g, or once g - g0 >= max_ext.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .boundaries import END, WORD\nfrom .simulator import Policy, TraceView\n\n\ndef entropy_fires(H: float, lam: float, beta: float) -> bool:\n    """AdaEDL/SVIP-style rule: stop when 1 - sqrt(beta * H) < lam."""\n    return 1.0 - math.sqrt(beta * max(float(H), 0.0)) < lam\n\n\ndef first_fire(view: TraceView, p: int, g: int, lam: float, beta: float) -> int | None:\n    """Smallest g\' in 1..g at which the entropy rule fired (reads H[p..p+g-1] only)."""\n    for k in range(1, g + 1):\n        if entropy_fires(view.feat("H", p + k - 1), lam, beta):\n            return k\n    return None\n\n\nclass NoDraft(Policy):\n    name = "NoDraft"\n\n    def initial_stop(self, view, p):\n        return True\n\n    def stop(self, view, p, g):\n        return True\n\n\nclass FixedK(Policy):\n    name = "FixedK"\n\n    def __init__(self, k: int):\n        self.k = int(k)\n\n    def initial_stop(self, view, p):\n        return self.k <= 0\n\n    def stop(self, view, p, g):\n        return g >= self.k\n\n    def params(self):\n        return dict(k=self.k)\n\n\nclass Entropy(Policy):\n    name = "Entropy"\n\n    def __init__(self, lam: float, beta: float = 1.0):\n        self.lam, self.beta = float(lam), float(beta)\n\n    def stop(self, view, p, g):\n        return entropy_fires(view.feat("H", p + g - 1), self.lam, self.beta)\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta)\n\n\nclass WordBoundarySoft(Policy):\n    """Stop when g >= min_g and the next draft distribution puts > tau mass on new-word tokens."""\n    name = "WordBoundarySoft"\n    lookahead = 1\n\n    def __init__(self, tau: float, min_g: int = 1):\n        self.tau, self.min_g = float(tau), int(min_g)\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and view.feat("p_newword", p + g) > self.tau\n\n    def params(self):\n        return dict(tau=self.tau, min_g=self.min_g)\n\n\nclass WordBoundaryHard(Policy):\n    """Corrected legacy word-boundary policy: stop when g >= min_g and the draft\'s next token\n    (argmax of q_{p+g}) starts with space / punctuation or is eos. Needs the vocabulary mask."""\n    name = "WordBoundaryHard"\n    lookahead = 1\n\n    def __init__(self, min_g: int, newword_mask: np.ndarray):\n        self.min_g = int(min_g)\n        self.mask = np.asarray(newword_mask, dtype=bool)\n\n    def stop(self, view, p, g):\n        if g < self.min_g:\n            return False\n        t = view.token(p + g)\n        return bool(t < len(self.mask) and self.mask[t])\n\n    def params(self):\n        return dict(min_g=self.min_g)\n\n\nclass _EntropyExtend(Policy):\n    """Entropy gate + linguistic condition at the lookahead position, with up to max_ext extensions."""\n    lookahead = 1\n\n    def __init__(self, lam: float, beta: float, max_ext: int):\n        self.lam, self.beta, self.max_ext = float(lam), float(beta), int(max_ext)\n\n    def cond(self, view: TraceView, p: int, g: int) -> bool:\n        raise NotImplementedError\n\n    def stop(self, view, p, g):\n        g0 = first_fire(view, p, g, self.lam, self.beta)\n        if g0 is None:\n            return False\n        return self.cond(view, p, g) or (g - g0) >= self.max_ext\n\n\nclass AksharaSafeEntropy(_EntropyExtend):\n    """Entropy rule, but only stop where the next token is unlikely to be a combining mark\n    (p_mark[p+g] < tau_m), i.e. the last akshara is probably complete."""\n    name = "AksharaSafeEntropy"\n\n    def __init__(self, lam: float, beta: float, tau_m: float, max_ext: int):\n        super().__init__(lam, beta, max_ext)\n        self.tau_m = float(tau_m)\n\n    def cond(self, view, p, g):\n        return view.feat("p_mark", p + g) < self.tau_m\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_m=self.tau_m, max_ext=self.max_ext)\n\n\nclass Hybrid(_EntropyExtend):\n    """Entropy rule, but only stop where the current word is probably complete (p_newword[p+g] > tau_w)."""\n    name = "Hybrid"\n\n    def __init__(self, lam: float, beta: float, tau_w: float, max_ext: int):\n        super().__init__(lam, beta, max_ext)\n        self.tau_w = float(tau_w)\n\n    def cond(self, view, p, g):\n        return view.feat("p_newword", p + g) > self.tau_w\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_w=self.tau_w, max_ext=self.max_ext)\n\n\nclass Oracle(Policy):\n    """Upper bound: drafts exactly the leading run of matches (can draft 0), never rejects."""\n    name = "Oracle"\n    is_oracle = True\n\n    def initial_stop(self, view, p):\n        return not view.gold("match", p)\n\n    def stop(self, view, p, g):\n        return not view.gold("match", p + g)\n\n\nclass GoldWordOracle(Policy):\n    """Upper bound for word-boundary stopping: stop (g >= min_g) when the *gold* cut after token\n    p+g-1 of the target is a word boundary. Free lookahead; needs ``attach_word_labels``."""\n    name = "GoldWordOracle"\n    is_oracle = True\n\n    def __init__(self, min_g: int = 1):\n        self.min_g = int(min_g)\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and view.gold("end_cut", p + g - 1) in (WORD, END)\n\n    def params(self):\n        return dict(min_g=self.min_g)\n\n\n# --------------------------------------------------------------------------- #\n# T6: morpheme policies\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass TokenTable:\n    """Per-vocabulary-id raw bytes and a word-start mask (space / punctuation / special)."""\n    token_bytes: list\n    word_start: np.ndarray\n\n    @classmethod\n    def from_tokenizer(cls, tok, masks=None) -> "TokenTable":\n        from .vocab import build_masks, token_bytes_list\n        masks = masks or build_masks(tok)\n        tb = token_bytes_list(tok, range(len(tok)))\n        return cls(tb, masks.starts_space | masks.starts_punct | masks.is_special)\n\n\ndef _complete_utf8(b: bytes) -> str:\n    for cut in range(len(b), max(len(b) - 4, -1), -1):\n        try:\n            return b[:cut].decode("utf-8")\n        except UnicodeDecodeError:\n            continue\n    return b.decode("utf-8", errors="ignore")\n\n\ndef current_word_prefix(view: TraceView, p: int, g: int, table: TokenTable, max_back: int = 32) -> str:\n    """Text of the current word after drafting g tokens: committed target tokens before p plus drafted\n    tokens p..p+g-1, scanning back to the last token that starts a word (that token\'s leading space\n    is stripped). Only complete UTF-8 characters are decoded."""\n    end = p + g - 1\n    parts = []\n    lo = max(0, end - max_back)\n    for idx in range(end, lo - 1, -1):\n        t = view.token(idx)\n        b = table.token_bytes[t] if t < len(table.token_bytes) else b""\n        parts.append(b)\n        if t < len(table.word_start) and table.word_start[t]:\n            break\n    raw = b"".join(reversed(parts)).lstrip(b" \\n\\t")\n    return _complete_utf8(raw)\n\n\ndef morph_prob(view, p: int, g: int, pm, table: TokenTable) -> float:\n    """pm(current word prefix) after drafting g tokens from p.\n\n    In the simulator the value at (trace, p, g) is the same for every config of every morph policy, so it\n    is cached on the trace object (keyed by the prefix model\'s unique ``uid``). It is computed the first time through the\n    causality-checked view, and (p, g) fixes exactly what the view allows, so reuse cannot leak\n    information. Live views (no ``trace``) are never cached."""\n    tr = getattr(view, "trace", None)\n    if tr is None:\n        return pm.prob(current_word_prefix(view, p, g, table))\n    cache = tr.__dict__.setdefault("_pm_cache", {})\n    key = (getattr(pm, "uid", id(pm)), p, g)\n    v = cache.get(key)\n    if v is None:\n        v = cache[key] = pm.prob(current_word_prefix(view, p, g, table))\n    return v\n\n\nclass MorphEntropy(_EntropyExtend):\n    """Entropy rule; stop where the causal prefix model predicts a morpheme boundary (pm > tau_morph).\n    If use_akshara (lookahead 1): the condition also holds when the word is complete\n    (p_newword[p+g] > 0.5), and additionally requires p_mark[p+g] < tau_m.\n    Without use_akshara the policy is causal (lookahead 0). Extends up to max_ext tokens."""\n    name = "MorphEntropy"\n\n    def __init__(self, lam: float, beta: float, tau_morph: float, use_akshara: bool, tau_m: float,\n                 max_ext: int, prefix_model, table: TokenTable):\n        super().__init__(lam, beta, max_ext)\n        self.tau_morph, self.use_akshara, self.tau_m = float(tau_morph), bool(use_akshara), float(tau_m)\n        self.pm, self.table = prefix_model, table\n        self.lookahead = 1 if self.use_akshara else 0\n\n    def cond(self, view, p, g):\n        ok = morph_prob(view, p, g, self.pm, self.table) > self.tau_morph\n        if self.use_akshara:\n            ok = (ok or view.feat("p_newword", p + g) > 0.5) and view.feat("p_mark", p + g) < self.tau_m\n        return ok\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_morph=self.tau_morph, use_akshara=self.use_akshara,\n                    tau_m=self.tau_m, max_ext=self.max_ext)\n\n\nclass MorphOnly(Policy):\n    """Pure morpheme policy (no entropy, causal): stop when g >= min_g and pm > tau_morph."""\n    name = "MorphOnly"\n\n    def __init__(self, tau_morph: float, min_g: int, prefix_model, table: TokenTable):\n        self.tau_morph, self.min_g = float(tau_morph), int(min_g)\n        self.pm, self.table = prefix_model, table\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and morph_prob(view, p, g, self.pm, self.table) > self.tau_morph\n\n    def params(self):\n        return dict(tau_morph=self.tau_morph, min_g=self.min_g)\n',
    'morph': '"""Morpheme labelling (PLAN T3, fixes B1): MorphTok loader, gold lexicon, CRF segmenter,\ntiered labeller, causal prefix-boundary model, per-token gold morph labels.\n\nConventions\n- All strings are NFC.\n- A word\'s morpheme boundaries are *internal* char offsets ``0 < b < len(word)``.\n- MorphTok segmentations include sandhi splitting; only entries whose pieces re-join to the word\n  are used. Words with several segmentations get the **union** of their boundaries.\n- Gold boundaries that fall inside an akshara (e.g. ``निकाल+ेगी``) are **snapped left** to the start\n  of that akshara (``निका|लेगी``); a boundary that would snap to 0 is dropped. (User decision.)\n- Words for labelling target text are maximal runs of characters that are neither whitespace nor\n  punctuation (consistent with ``boundaries.is_word_boundary``).\n"""\nfrom __future__ import annotations\n\nimport functools\nimport gzip\nimport hashlib\nimport pickle\nimport random\nimport re\nimport unicodedata\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\nfrom .boundaries import END, MIDWORD, WORD, akshara_units\n\nDEVANAGARI_RE = re.compile(r"[ऀ-ॿ]")\n\n\ndef nfc(s: str) -> str:\n    return unicodedata.normalize("NFC", s)\n\n\ndef is_devanagari_word(w: str) -> bool:\n    return bool(DEVANAGARI_RE.search(w))\n\n\ndef akshara_boundaries(word: str) -> set[int]:\n    """Internal akshara-unit boundaries of ``word``."""\n    return {e for _, e in akshara_units(word)[:-1]}\n\n\ndef snap_to_akshara(word: str, bounds: Iterable[int]) -> tuple[set[int], int]:\n    """Snap each boundary inside an akshara left to the akshara start. Returns (bounds, n_snapped)."""\n    units = akshara_units(word)\n    out, snapped = set(), 0\n    for b in bounds:\n        for s, e in units:\n            if s < b < e:\n                snapped += 1\n                b = s\n                break\n        if 0 < b < len(word):\n            out.add(b)\n    return out, snapped\n\n\ndef seg_boundaries(seg: str) -> set[int]:\n    out, pos = set(), 0\n    for p in seg.split("+")[:-1]:\n        pos += len(p)\n        out.add(pos)\n    return out\n\n\ndef iter_words(text: str) -> list[tuple[int, int]]:\n    """(start, end) char spans of words: maximal runs of non-space, non-punctuation chars."""\n    spans, i, n = [], 0, len(text)\n    while i < n:\n        if text[i].isspace() or unicodedata.category(text[i]).startswith("P"):\n            i += 1\n            continue\n        j = i\n        while j < n and not (text[j].isspace() or unicodedata.category(text[j]).startswith("P")):\n            j += 1\n        spans.append((i, j))\n        i = j\n    return spans\n\n\n# --------------------------------------------------------------------------- #\n# 1. MorphTok loader\n# --------------------------------------------------------------------------- #\n\ndef load_morphtok(path) -> tuple["object", dict]:\n    """Read MorphTok csv (columns word, segmentation, language), NFC-normalise, mark entries whose\n    pieces re-join to the word. Returns (DataFrame with column ``kept``, info dict)."""\n    import pandas as pd\n    path = Path(path)\n    df = pd.read_csv(path, dtype=str, keep_default_na=False)\n    expected = ["word", "segmentation", "language"]\n    if list(df.columns) != expected:\n        raise ValueError(f"unexpected MorphTok columns {list(df.columns)}; expected {expected}")\n    df["word"] = df["word"].map(nfc)\n    df["segmentation"] = df["segmentation"].map(nfc)\n    df["kept"] = df["segmentation"].str.replace("+", "", regex=False) == df["word"]\n    dropped = df[~df["kept"]]\n    info = dict(path=str(path.name), sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                rows=int(len(df)), unique_words=int(df["word"].nunique()),\n                kept_rows=int(df["kept"].sum()), dropped_rows=int((~df["kept"]).sum()),\n                dropped_examples=[f"{w} -> {s}" for w, s in zip(dropped["word"][:10], dropped["segmentation"][:10])],\n                languages=dict(Counter(df["language"])))\n    return df, info\n\n\n# --------------------------------------------------------------------------- #\n# 2. Gold lexicon\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass GoldLexicon:\n    bounds: dict[str, frozenset] = field(default_factory=dict)\n    stats: dict = field(default_factory=dict)\n\n    @classmethod\n    def from_dataframe(cls, df) -> "GoldLexicon":\n        raw: dict[str, set] = defaultdict(set)\n        nseg: Counter = Counter()\n        for w, s in zip(df.loc[df["kept"], "word"], df.loc[df["kept"], "segmentation"]):\n            raw[w] |= seg_boundaries(s)\n            nseg[w] += 1\n        out, snapped, total = {}, 0, 0\n        for w, b in raw.items():\n            total += len(b)\n            sb, k = snap_to_akshara(w, b)\n            snapped += k\n            out[w] = frozenset(sb)\n        stats = dict(words=len(out), words_with_multiple_segmentations=sum(1 for v in nseg.values() if v > 1),\n                     raw_internal_boundaries=total, boundaries_inside_akshara_snapped=snapped,\n                     final_internal_boundaries=sum(len(v) for v in out.values()))\n        return cls(out, stats)\n\n    def __contains__(self, w: str) -> bool:\n        return w in self.bounds\n\n    def get(self, w: str):\n        return self.bounds.get(w)\n\n\n# --------------------------------------------------------------------------- #\n# 3. CRF segmenter\n# --------------------------------------------------------------------------- #\n\ndef _char_features(word: str, i: int, aks_start: set[int], aks_end: set[int], pos_in_aks: list[int]) -> dict:\n    ch = word[i]\n    n = len(word)\n    f = {\n        "bias": 1.0, "c": ch, "cat": unicodedata.category(ch),\n        "aks_start": i in aks_start, "aks_end": (i + 1) in aks_end, "pos_in_aks": pos_in_aks[i],\n        "len": min(n, 15), "from_start": min(i, 8), "from_end": min(n - 1 - i, 8),\n    }\n    for d in range(-3, 4):  # char n-grams in a ±3 window\n        j = i + d\n        f[f"c[{d}]"] = word[j] if 0 <= j < n else ("<s>" if j < 0 else "</s>")\n    for a, b in [(-2, 0), (-1, 1), (0, 2), (1, 3), (-3, 0), (0, 3)]:\n        f[f"ng[{a}:{b}]"] = "".join(word[j] if 0 <= j < n else "_" for j in range(i + a, i + b + 1))\n    return f\n\n\ndef word2features(word: str) -> list[dict]:\n    units = akshara_units(word)\n    aks_start = {s for s, _ in units}\n    aks_end = {e for _, e in units}\n    pos_in_aks = [0] * len(word)\n    for s, e in units:\n        for k in range(s, e):\n            pos_in_aks[k] = min(k - s, 4)\n    return [_char_features(word, i, aks_start, aks_end, pos_in_aks) for i in range(len(word))]\n\n\ndef word2labels(word: str, bounds: Iterable[int]) -> list[str]:\n    """Tag of char i is \'B\' if there is a boundary after it (offset i+1)."""\n    b = set(bounds)\n    return ["B" if (i + 1) in b else "O" for i in range(len(word))]\n\n\nclass CRFSegmenter:\n    """Character-level CRF predicting \'boundary after this char\'; predictions are post-processed so\n    boundaries only fall on akshara boundaries."""\n\n    def __init__(self, c1: float = 0.1, c2: float = 0.05, max_iterations: int = 150):\n        import sklearn_crfsuite\n        self.crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=c1, c2=c2, max_iterations=max_iterations,\n                                        all_possible_transitions=True)\n\n    def fit(self, words: Sequence[str], bounds: Sequence[Iterable[int]]) -> "CRFSegmenter":\n        self.crf.fit([word2features(w) for w in words], [word2labels(w, b) for w, b in zip(words, bounds)])\n        return self\n\n    def predict_raw(self, word: str) -> set[int]:\n        tags = self.crf.predict_single(word2features(word))\n        return {i + 1 for i, t in enumerate(tags) if t == "B" and i + 1 < len(word)}\n\n    def predict(self, word: str) -> set[int]:\n        if not word:\n            return set()\n        return self.predict_raw(word) & akshara_boundaries(word)\n\n    def predict_many(self, words: Sequence[str]) -> list[set[int]]:\n        tags = self.crf.predict([word2features(w) for w in words])\n        out = []\n        for w, ts in zip(words, tags):\n            raw = {i + 1 for i, t in enumerate(ts) if t == "B" and i + 1 < len(w)}\n            out.append(raw & akshara_boundaries(w))\n        return out\n\n\ndef split_by_type(words: Sequence[str], frac_test: float = 0.1, seed: int = 0) -> tuple[list[str], list[str]]:\n    ws = sorted(set(words))\n    random.Random(seed).shuffle(ws)\n    k = int(round(len(ws) * frac_test))\n    return ws[k:], ws[:k]\n\n\ndef boundary_prf(gold: Sequence[set], pred: Sequence[set]) -> dict:\n    tp = sum(len(g & p) for g, p in zip(gold, pred))\n    np_ = sum(len(p) for p in pred)\n    ng = sum(len(g) for g in gold)\n    P = tp / np_ if np_ else float("nan")\n    R = tp / ng if ng else float("nan")\n    F = 2 * P * R / (P + R) if np_ and ng and (P + R) else float("nan")\n    exact = float(np.mean([g == p for g, p in zip(gold, pred)])) if gold else float("nan")\n    return dict(precision=P, recall=R, f1=F, tp=tp, n_pred=np_, n_gold=ng, word_exact_match=exact)\n\n\ndef train_and_eval_crf(lex: GoldLexicon, seed: int = 0, **crf_kw) -> tuple[CRFSegmenter, dict]:\n    """Train on 90% of word types, evaluate boundary P/R/F1 on the held-out 10%, then return a model\n    trained on the 90% split (the held-out metrics describe exactly the saved model)."""\n    import time\n    train, test = split_by_type(list(lex.bounds), 0.1, seed)\n    t0 = time.time()\n    seg = CRFSegmenter(**crf_kw).fit(train, [lex.bounds[w] for w in train])\n    t_fit = time.time() - t0\n    gold = [set(lex.bounds[w]) for w in test]\n    pred = seg.predict_many(test)\n    raw = [seg.predict_raw(w) for w in test]\n    m = boundary_prf(gold, pred)\n    m_raw = boundary_prf(gold, raw)\n    # baseline: no boundaries at all / every akshara boundary\n    m_all = boundary_prf(gold, [akshara_boundaries(w) for w in test])\n    info = dict(seed=seed, n_train_types=len(train), n_test_types=len(test), fit_seconds=t_fit,\n                crf_params=dict(c1=seg.crf.c1, c2=seg.crf.c2, max_iterations=seg.crf.max_iterations),\n                heldout=m, heldout_before_akshara_postprocess=m_raw,\n                baseline_every_akshara_boundary=m_all,\n                n_raw_predictions_removed_by_postprocess=sum(len(r - p) for r, p in zip(raw, pred)))\n    return seg, info\n\n\n# --------------------------------------------------------------------------- #\n# 4. Tiered labeller\n# --------------------------------------------------------------------------- #\n\nclass TieredLabeler:\n    """label(word) -> (boundaries, tier) with tiers gold -> crf -> none (none: non-Devanagari)."""\n\n    def __init__(self, lex: GoldLexicon, crf: CRFSegmenter | None):\n        self.lex, self.crf = lex, crf\n        self._cache: dict[str, tuple[frozenset, str]] = {}\n\n    def label(self, word: str) -> tuple[frozenset, str]:\n        word = nfc(word)\n        r = self._cache.get(word)\n        if r is not None:\n            return r\n        if word in self.lex:\n            r = (self.lex.get(word), "gold")\n        elif is_devanagari_word(word) and self.crf is not None:\n            r = (frozenset(self.crf.predict(word)), "crf")\n        else:\n            r = (frozenset(), "none")\n        self._cache[word] = r\n        return r\n\n    def label_text(self, text: str) -> list[tuple[int, int, frozenset, str]]:\n        return [(s, e, *self.label(text[s:e])) for s, e in iter_words(text)]\n\n    def coverage(self, texts: Iterable[str]) -> dict:\n        """Per-tier coverage over word tokens and word types."""\n        tok, types = Counter(), {}\n        for t in texts:\n            for s, e, _, tier in self.label_text(nfc(t)):\n                tok[tier] += 1\n                types[t[s:e]] = tier\n        typ = Counter(types.values())\n        nt, ny = sum(tok.values()), sum(typ.values())\n        return dict(word_tokens=nt, word_types=ny,\n                    tokens={k: dict(n=tok[k], frac=tok[k] / nt if nt else float("nan")) for k in ("gold", "crf", "none")},\n                    types={k: dict(n=typ[k], frac=typ[k] / ny if ny else float("nan")) for k in ("gold", "crf", "none")})\n\n\n# --------------------------------------------------------------------------- #\n# 5. Causal prefix-boundary model\n# --------------------------------------------------------------------------- #\n\nclass PrefixBoundaryModel:\n    """P(internal morpheme boundary right after ``prefix`` | prefix) with backoff:\n    full prefix -> last 3 aksharas -> last 2 -> last 1 -> global rate.\n    A level is used only if its count >= ``min_count``. Built from akshara-aligned proper prefixes\n    of labelled words (MorphTok + dev target outputs only; never devtest)."""\n\n    LEVELS = ("full", "last3", "last2", "last1")\n\n    def __init__(self, min_count: int = 5):\n        import uuid\n        self.uid = uuid.uuid4().hex   # identifies this model in per-trace caches (never reused)\n        self.min_count = min_count\n        self.pos: dict[str, Counter] = {k: Counter() for k in self.LEVELS}\n        self.tot: dict[str, Counter] = {k: Counter() for k in self.LEVELS}\n        self.g_pos = 0.0\n        self.g_tot = 0.0\n        self.sources: dict = {}\n\n    @staticmethod\n    def _keys(prefix: str) -> dict[str, str]:\n        units = akshara_units(prefix)\n        u = [prefix[s:e] for s, e in units]\n        return {"full": prefix, "last3": "".join(u[-3:]), "last2": "".join(u[-2:]), "last1": "".join(u[-1:])}\n\n    def add_word(self, word: str, bounds: Iterable[int], weight: float = 1.0) -> None:\n        b = set(bounds)\n        for c in sorted(akshara_boundaries(word)):          # proper, akshara-aligned prefixes\n            y = c in b\n            for lvl, key in self._keys(word[:c]).items():\n                self.tot[lvl][key] += weight\n                if y:\n                    self.pos[lvl][key] += weight\n            self.g_tot += weight\n            self.g_pos += weight * y\n\n    def fit(self, weighted_words: Iterable[tuple[str, Iterable[int], float]], sources: dict | None = None):\n        for w, b, wt in weighted_words:\n            self.add_word(w, b, wt)\n        self.sources = sources or {}\n        self.prob.cache_clear()\n        import uuid\n        self.uid = uuid.uuid4().hex   # contents changed: invalidate per-trace caches\n        return self\n\n    @property\n    def global_rate(self) -> float:\n        return self.g_pos / self.g_tot if self.g_tot else 0.0\n\n    def level_used(self, prefix: str) -> str:\n        if not prefix:\n            return "global"\n        for lvl, key in self._keys(nfc(prefix)).items():\n            if self.tot[lvl][key] >= self.min_count:\n                return lvl\n        return "global"\n\n    @functools.lru_cache(maxsize=1_000_000)\n    def prob(self, prefix: str) -> float:\n        if not prefix:\n            return self.global_rate\n        for lvl, key in self._keys(nfc(prefix)).items():\n            n = self.tot[lvl].get(key, 0)\n            if n >= self.min_count:\n                return float(self.pos[lvl].get(key, 0) / n)\n        return self.global_rate\n\n    def __hash__(self):  # lru_cache on a method keys on self\n        return id(self)\n\n    def save(self, path) -> None:\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        state = dict(min_count=self.min_count, pos={k: dict(v) for k, v in self.pos.items()},\n                     tot={k: dict(v) for k, v in self.tot.items()}, g_pos=self.g_pos, g_tot=self.g_tot,\n                     sources=self.sources)\n        with gzip.open(path, "wb") as fh:\n            pickle.dump(state, fh, protocol=4)\n\n    @classmethod\n    def load(cls, path) -> "PrefixBoundaryModel":\n        with gzip.open(path, "rb") as fh:\n            s = pickle.load(fh)\n        return cls.from_state(s)\n\n    @classmethod\n    def from_state(cls, s: dict) -> "PrefixBoundaryModel":\n        m = cls(s["min_count"])\n        m.pos = {k: Counter(v) for k, v in s["pos"].items()}\n        m.tot = {k: Counter(v) for k, v in s["tot"].items()}\n        m.g_pos, m.g_tot, m.sources = s["g_pos"], s["g_tot"], s.get("sources", {})\n        return m\n\n\ndef build_prefix_model(lex: GoldLexicon, labeler: TieredLabeler, dev_texts: Iterable[str],\n                       min_count: int = 5) -> PrefixBoundaryModel:\n    """MorphTok words (weight 1 each) + every Devanagari word of the dev target outputs labelled by\n    the tiered labeller, weighted by its frequency in dev. Never pass devtest texts here."""\n    freq = Counter()\n    for t in dev_texts:\n        t = nfc(t)\n        for s, e in iter_words(t):\n            freq[t[s:e]] += 1\n    items = [(w, b, 1.0) for w, b in lex.bounds.items() if is_devanagari_word(w)]\n    tiers = Counter()\n    for w, c in freq.items():\n        b, tier = labeler.label(w)\n        if tier == "none":\n            continue\n        tiers[tier] += c\n        items.append((w, b, float(c)))\n    src = dict(morphtok_words=len(lex.bounds), dev_word_types=len(freq), dev_word_tokens_by_tier=dict(tiers),\n               min_count=min_count)\n    return PrefixBoundaryModel(min_count).fit(items, src)\n\n\n# --------------------------------------------------------------------------- #\n# 6. Per-token gold morph labels (analysis only)\n# --------------------------------------------------------------------------- #\n\ndef token_morph_labels(word_labels: dict, text: str, labeler: TieredLabeler) -> dict[str, np.ndarray]:\n    """``word_labels`` is the output of ``boundaries.token_labels`` for the same sequence and\n    ``text`` its decoded text. Returns\n      morph_boundary_after[t]: the cut after t is a word boundary/end or an internal morpheme boundary\n      morph_tier[t]: tier (\'gold\'/\'crf\'/\'none\') of the word containing token t (\'none\' outside words)."""\n    end_cut, spans = word_labels["end_cut"], word_labels["char_span"]\n    n = len(end_cut)\n    words = labeler.label_text(text)\n    char_word = np.full(len(text) + 1, -1, dtype=np.int64)\n    for k, (s, e, _, _) in enumerate(words):\n        char_word[s:e] = k\n    after = np.zeros(n, dtype=bool)\n    tier = np.full(n, "none", dtype=object)\n    for t in range(n):\n        cs, ce = int(spans[t][0]), int(spans[t][1])\n        # tier: first word char inside the token\'s span\n        for c in range(cs, max(ce, cs + 1)):\n            if c < len(text) and char_word[c] >= 0:\n                tier[t] = words[char_word[c]][3]\n                break\n        if end_cut[t] in (WORD, END):\n            after[t] = True\n            continue\n        if end_cut[t] != MIDWORD:              # mid-akshara / mid-UTF-8 cuts are never morph boundaries\n            continue\n        k = char_word[ce - 1] if 0 < ce <= len(text) else -1\n        if k >= 0 and char_word[ce] == k:      # cut strictly inside word k\n            s = words[k][0]\n            after[t] = (ce - s) in words[k][2]\n    return dict(morph_boundary_after=after, morph_tier=tier)\n\n\ndef attach_morph_labels(traces, labeler: TieredLabeler) -> None:\n    """Add morph_boundary_after / morph_tier to traces that already have word labels (and ``text``)."""\n    for t in traces:\n        t.labels.update(token_morph_labels(t.labels, t.labels["text"], labeler))\n',
    'stats': '"""Bootstrap statistics over sentences (PLAN T7.4).\n\nSpeed-up on a resample = sum(emitted) / sum(cost) over the resampled sentences (ratio of sums, the\nsame estimator as the point value). Paired comparisons use the same resampled indices for both\npolicies, which must have been simulated on the same sentence list.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef _idx(n: int, n_boot: int, seed: int) -> np.ndarray:\n    return np.random.default_rng(seed).integers(0, n, size=(n_boot, n))\n\n\ndef bootstrap_ci(emitted, cost, n_boot: int = 1000, seed: int = 0, level: float = 0.95) -> dict:\n    e, c = np.asarray(emitted, float), np.asarray(cost, float)\n    I = _idx(len(e), n_boot, seed)\n    s = e[I].sum(1) / c[I].sum(1)\n    a = (1 - level) / 2\n    return dict(point=float(e.sum() / c.sum()), lo=float(np.quantile(s, a)), hi=float(np.quantile(s, 1 - a)),\n                n_boot=n_boot, seed=seed)\n\n\ndef paired_bootstrap(emitted_a, cost_a, emitted_b, cost_b, n_boot: int = 1000, seed: int = 0,\n                     level: float = 0.95) -> dict:\n    """Difference speedup(a) - speedup(b) with a paired bootstrap over sentences.\n    p_better = fraction of resamples where the difference is > 0."""\n    ea, ca = np.asarray(emitted_a, float), np.asarray(cost_a, float)\n    eb, cb = np.asarray(emitted_b, float), np.asarray(cost_b, float)\n    if not (len(ea) == len(ca) == len(eb) == len(cb)):\n        raise ValueError("paired bootstrap needs per-sentence arrays of equal length")\n    I = _idx(len(ea), n_boot, seed)\n    d = ea[I].sum(1) / ca[I].sum(1) - eb[I].sum(1) / cb[I].sum(1)\n    a = (1 - level) / 2\n    return dict(diff=float(ea.sum() / ca.sum() - eb.sum() / cb.sum()), lo=float(np.quantile(d, a)),\n                hi=float(np.quantile(d, 1 - a)), p_better=float((d > 0).mean()), n_boot=n_boot, seed=seed)\n\n\ndef bootstrap_stat_by_sentence(groups: list, fn, n_boot: int = 1000, seed: int = 0, level: float = 0.95) -> dict:\n    """Generic sentence-level bootstrap: ``groups`` is a list of per-sentence objects, ``fn`` maps a\n    list of them to a scalar."""\n    I = _idx(len(groups), n_boot, seed)\n    vals = np.array([fn([groups[i] for i in row]) for row in I], float)\n    n_valid = int(np.isfinite(vals).sum())\n    vals = vals[np.isfinite(vals)]\n    a = (1 - level) / 2\n    lo = float(np.quantile(vals, a)) if n_valid else float("nan")      # e.g. constant variable\n    hi = float(np.quantile(vals, 1 - a)) if n_valid else float("nan")\n    return dict(point=float(fn(groups)), lo=lo, hi=hi, n_boot=n_boot, n_valid=n_valid, seed=seed)\n',
    'tuning': '"""Grid search on FLORES+ dev, evaluation on devtest (PLAN T7, fixes B7, B8).\n\nEvery policy family gets a dev grid; selection = best dev speed-up with lookahead charged and the\nmeasured verify cost. Grids (PLAN T5 table + user decisions of 2026-11-02 for the parts the plan\nleaves open):\n  - "around the entropy best": lam = Entropy\'s dev-best lam +- 5 steps of Entropy\'s lam grid,\n    beta in all of Entropy\'s betas.\n  - MorphEntropy: tau_morph in {0.1..0.6}, use_akshara in {F, T}, tau_m in {0.3, 0.5} (only with\n    use_akshara), max_ext in {0..3}. MorphOnly: tau_morph in linspace(0.05, 0.95, 19), min_g in {1,2,3}.\n"""\nfrom __future__ import annotations\n\nimport itertools\nimport json\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable\n\nimport numpy as np\n\nfrom .boundaries import END, MIDAKS, MIDUTF8, WORD\nfrom .policies import (AksharaSafeEntropy, Entropy, FixedK, GoldWordOracle, Hybrid, MorphEntropy, MorphOnly,\n                       NoDraft, Oracle, TokenTable, WordBoundaryHard, WordBoundarySoft)\nfrom .simulator import GAMMA_MAX, simulate, summarize, unit_verify\nfrom .stats import bootstrap_ci, paired_bootstrap\n\nENTROPY_LAMS = [float(x) for x in np.round(np.linspace(-1.5, 0.95, 40), 3)]\nENTROPY_BETAS = [0.5, 1.0, 2.0, 4.0]\nLAM_NBHD = 5\n\nFAMILIES = ["NoDraft", "FixedK", "Entropy", "WordBoundarySoft", "WordBoundaryHard", "AksharaSafeEntropy",\n            "Hybrid", "MorphEntropy", "MorphOnly", "Oracle", "GoldWordOracle",\n            "Hybrid_ext1", "MorphEntropy_ext1"]\n# Constrained variants "<Family>_ext1" (user decision 2026-11-02): same class and grid, restricted to max_ext >= 1 so the\n# linguistic condition is actually consulted (max_ext = 0 means "stop whenever entropy fires").\nCONSTRAINED = {"Hybrid_ext1": "Hybrid", "MorphEntropy_ext1": "MorphEntropy"}\n\n\ndef base_family(family: str) -> str:\n    return CONSTRAINED.get(family, family)\n\n\ndef reduction_note(family: str, params: dict) -> str:\n    """Human-readable note when a selected config is equivalent to a simpler policy."""\n    fam = base_family(family)\n    if fam == "MorphEntropy" and params.get("max_ext") == 0 and not params.get("use_akshara"):\n        return "reduces to Entropy (max_ext=0: morph model never consulted)"\n    if fam == "MorphEntropy" and params.get("max_ext") == 0:\n        return "max_ext=0: stops whenever entropy fires; only adds a charged lookahead"\n    if fam == "Hybrid" and params.get("max_ext") == 0:\n        return "max_ext=0: Entropy + charged lookahead (word condition never changes the decision)"\n    return ""\nORACLES = {"Oracle", "GoldWordOracle"}\n\n\n@dataclass\nclass Context:\n    """Everything policies need besides scalar params."""\n    newword_mask: np.ndarray | None = None      # VocabMasks.starts_newword\n    prefix_model: object = None                 # morph.PrefixBoundaryModel\n    table: TokenTable | None = None\n    entropy_best: dict = field(default_factory=dict)   # filled after tuning Entropy\n\n\ndef lam_neighbourhood(best_lam: float, k: int = LAM_NBHD) -> list[float]:\n    i = int(np.argmin(np.abs(np.array(ENTROPY_LAMS) - best_lam)))\n    return ENTROPY_LAMS[max(0, i - k): i + k + 1]\n\n\ndef grid(family: str, ctx: Context) -> list[dict]:\n    if family in CONSTRAINED:\n        return [p for p in grid(CONSTRAINED[family], ctx) if p["max_ext"] >= 1]\n    lb = None\n    if family in ("AksharaSafeEntropy", "Hybrid", "MorphEntropy"):\n        if not ctx.entropy_best:\n            raise RuntimeError("tune Entropy before " + family)\n        lb = list(itertools.product(lam_neighbourhood(ctx.entropy_best["lam"]), ENTROPY_BETAS))\n    if family == "NoDraft" or family == "Oracle":\n        return [{}]\n    if family == "FixedK":\n        return [dict(k=k) for k in range(1, 9)]\n    if family == "Entropy":\n        return [dict(lam=l, beta=b) for b in ENTROPY_BETAS for l in ENTROPY_LAMS]\n    if family == "WordBoundarySoft":\n        return [dict(tau=float(t), min_g=m) for t in np.round(np.linspace(0.05, 0.95, 19), 3) for m in (1, 2, 3)]\n    if family == "WordBoundaryHard":\n        return [dict(min_g=m) for m in (1, 2, 3)]\n    if family == "AksharaSafeEntropy":\n        return [dict(lam=l, beta=b, tau_m=t, max_ext=e) for (l, b) in lb for t in (0.1, 0.3, 0.5, 0.7) for e in (1, 2, 3)]\n    if family == "Hybrid":\n        return [dict(lam=l, beta=b, tau_w=t, max_ext=e) for (l, b) in lb for t in (0.2, 0.4, 0.6, 0.8) for e in (0, 1, 2, 3)]\n    if family == "MorphEntropy":\n        out = []\n        for (l, b) in lb:\n            for tm in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6):\n                for e in (0, 1, 2, 3):\n                    out.append(dict(lam=l, beta=b, tau_morph=tm, use_akshara=False, tau_m=0.5, max_ext=e))\n                    for tau_m in (0.3, 0.5):\n                        out.append(dict(lam=l, beta=b, tau_morph=tm, use_akshara=True, tau_m=tau_m, max_ext=e))\n        return out\n    if family == "MorphOnly":\n        return [dict(tau_morph=float(t), min_g=m) for t in np.round(np.linspace(0.05, 0.95, 19), 3) for m in (1, 2, 3)]\n    if family == "GoldWordOracle":\n        return [dict(min_g=m) for m in (1, 2, 3)]\n    raise KeyError(family)\n\n\ndef make_policy(family: str, params: dict, ctx: Context):\n    family = base_family(family)\n    p = dict(params)\n    if family == "NoDraft":\n        return NoDraft()\n    if family == "FixedK":\n        return FixedK(**p)\n    if family == "Entropy":\n        return Entropy(**p)\n    if family == "WordBoundarySoft":\n        return WordBoundarySoft(**p)\n    if family == "WordBoundaryHard":\n        return WordBoundaryHard(p["min_g"], ctx.newword_mask)\n    if family == "AksharaSafeEntropy":\n        return AksharaSafeEntropy(**p)\n    if family == "Hybrid":\n        return Hybrid(**p)\n    if family == "MorphEntropy":\n        return MorphEntropy(**p, prefix_model=ctx.prefix_model, table=ctx.table)\n    if family == "MorphOnly":\n        return MorphOnly(**p, prefix_model=ctx.prefix_model, table=ctx.table)\n    if family == "Oracle":\n        return Oracle()\n    if family == "GoldWordOracle":\n        return GoldWordOracle(**p)\n    raise KeyError(family)\n\n\ndef cap_for(family: str, params: dict) -> int:\n    """Fixed-k drafts exactly k (legacy: gamma_max = k); every dynamic policy uses GAMMA_MAX = 8."""\n    return int(params["k"]) if family == "FixedK" else GAMMA_MAX\n\n\n_W: dict = {}\n\n\ndef _worker_init(traces, c, verify, ctx, family):\n    _W.update(traces=traces, c=c, verify=verify, ctx=ctx, family=family)\n\n\ndef _grid_rows(params_list):\n    fam, traces, c, verify, ctx = _W["family"], _W["traces"], _W["c"], _W["verify"], _W["ctx"]\n    rows = []\n    for params in params_list:\n        pol = make_policy(fam, params, ctx)\n        r = simulate(traces, pol, cap_for(fam, params), c, verify, charge_lookahead=True, record_rounds=False)\n        rows.append(dict(family=fam, **params, lookahead=pol.lookahead, **summarize(r)))\n    return rows\n\n\ndef run_grid(family: str, traces, c: float, verify: Callable, ctx: Context, progress: bool = False,\n             n_jobs: int | None = None, parallel_min: int = 100):\n    """Simulate every config of ``family`` on ``traces``. Grids with >= parallel_min configs run in\n    ``n_jobs`` worker processes (default: all cores but one); results are identical to serial."""\n    import os\n    import pandas as pd\n    t0 = time.time()\n    configs = grid(family, ctx)\n    n_jobs = n_jobs if n_jobs is not None else max(1, (os.cpu_count() or 2) - 1)\n    if n_jobs > 1 and len(configs) >= parallel_min:\n        from concurrent.futures import ProcessPoolExecutor\n        chunks = [configs[i::n_jobs * 4] for i in range(n_jobs * 4)]\n        with ProcessPoolExecutor(n_jobs, initializer=_worker_init, initargs=(traces, c, verify, ctx, family)) as ex:\n            parts = list(ex.map(_grid_rows, chunks))\n        order = {json.dumps(p, sort_keys=True): i for i, p in enumerate(configs)}\n        rows = sorted((r for part in parts for r in part),\n                      key=lambda r: order[json.dumps({k: r[k] for k in configs[0]}, sort_keys=True)])\n    else:\n        _worker_init(traces, c, verify, ctx, family)\n        rows = _grid_rows(configs)\n    df = pd.DataFrame(rows)\n    if progress:\n        print(f"{family:20s} {len(df):5d} configs  {time.time() - t0:6.1f}s  best dev speed-up {df.speedup.max():.4f}")\n    return df\n\n\ndef select(df) -> dict:\n    """Params of the best dev speed-up (ties -> first in grid order)."""\n    best = df.loc[df.speedup.idxmax()]\n    drop = {"family", "lookahead", "acceptance_rate", "alpha_hat", "tokens_per_verif", "mean_draft_len", "speedup"}\n    out = {}\n    for k, v in best.items():\n        if k in drop or (isinstance(v, float) and np.isnan(v)):\n            continue\n        out[k] = v.item() if hasattr(v, "item") else v\n    if "k" in out or "min_g" in out or "max_ext" in out:\n        for key in ("k", "min_g", "max_ext"):\n            if key in out:\n                out[key] = int(out[key])\n    if "use_akshara" in out:\n        out["use_akshara"] = bool(out["use_akshara"])\n    return out\n\n\ndef tune_all(dev, c: float, verify: Callable, ctx: Context, out_dir=None, families=FAMILIES,\n             progress: bool = True, n_jobs: int | None = None) -> dict:\n    """Grid search every family on dev; saves grid_{family}.csv; returns {family: params}."""\n    out_dir = Path(out_dir) if out_dir else None\n    order = ["FixedK", "Entropy"] + [f for f in families if f not in ("FixedK", "Entropy")]\n    selected = {}\n    for fam in order:\n        if fam not in families and fam not in ("FixedK", "Entropy"):\n            continue\n        df = run_grid(fam, dev, c, verify, ctx, progress, n_jobs=n_jobs)\n        if out_dir:\n            df.to_csv(out_dir / f"grid_{fam}.csv", index=False)\n        selected[fam] = select(df)\n        if fam == "Entropy":\n            ctx.entropy_best = dict(selected[fam])\n    return selected\n\n\n# --------------------------------------------------------------------------- #\n# evaluation on devtest\n# --------------------------------------------------------------------------- #\n\ndef _ending_fracs(records, traces_by_id) -> dict:\n    """Fractions of non-empty drafts whose last drafted index p+g-1 ends at a gold word boundary, inside\n    an akshara (incl. mid-UTF-8), or at a gold morpheme boundary (labels of the target sequence)."""\n    n = w = a = m = 0\n    has_m = True\n    for sid, p, g, acc, _ in records:\n        if g == 0:\n            continue\n        L = traces_by_id[sid].labels\n        cut = L["end_cut"][p + g - 1]\n        n += 1\n        w += cut in (WORD, END)\n        a += cut in (MIDAKS, MIDUTF8)\n        if "morph_boundary_after" in L:\n            m += bool(L["morph_boundary_after"][p + g - 1])\n        else:\n            has_m = False\n    return dict(frac_drafts_ending_at_gold_word_boundary=w / n if n else float("nan"),\n                frac_ending_mid_akshara=a / n if n else float("nan"),\n                frac_ending_at_morph_boundary=(m / n if n else float("nan")) if has_m else float("nan"))\n\n\ndef evaluate(test, selected: dict, c: float, verify: Callable, ctx: Context, n_boot: int = 1000, seed: int = 0):\n    """Main table on devtest. Returns (DataFrame, per-policy SimResults charged)."""\n    import pandas as pd\n    by_id = {t.id: t for t in test}\n    sims, rows = {}, []\n    for fam, params in selected.items():\n        pol = make_policy(fam, params, ctx)\n        cap = cap_for(fam, params)\n        ch = simulate(test, pol, cap, c, verify, charge_lookahead=True)\n        un = simulate(test, pol, cap, c, verify, charge_lookahead=False, record_rounds=False)\n        uv = simulate(test, pol, cap, c, unit_verify, charge_lookahead=True, record_rounds=False)\n        sims[fam] = ch\n        m = summarize(ch)\n        ci = bootstrap_ci(ch.sent_emitted, ch.sent_cost, n_boot, seed)\n        has_labels = all("end_cut" in t.labels for t in test)\n        fr = _ending_fracs(ch.records, by_id) if has_labels else {}\n        rows.append(dict(policy=fam, note=reduction_note(fam, params),\n                         params=json.dumps(params, ensure_ascii=False), lookahead=pol.lookahead,\n                         speedup_charged=m["speedup"], CI_low=ci["lo"], CI_high=ci["hi"],\n                         speedup_uncharged=summarize(un)["speedup"], speedup_unit_verify=summarize(uv)["speedup"],\n                         acceptance_rate=m["acceptance_rate"], alpha_hat=m["alpha_hat"],\n                         tokens_per_verif=m["tokens_per_verif"], mean_draft_len=m["mean_draft_len"], **fr))\n    df = pd.DataFrame(rows)\n    if "Entropy" in sims:\n        e = sims["Entropy"]\n        d = [paired_bootstrap(sims[f].sent_emitted, sims[f].sent_cost, e.sent_emitted, e.sent_cost, n_boot, seed)\n             for f in df.policy]\n        df["diff_vs_entropy"] = [x["diff"] for x in d]\n        df["diff_CI_low"] = [x["lo"] for x in d]\n        df["diff_CI_high"] = [x["hi"] for x in d]\n        df["p_better"] = [x["p_better"] for x in d]\n    if "FixedK" in sims and "Oracle" in sims:\n        fb = df.set_index("policy").loc["FixedK", "speedup_charged"]\n        orc = df.set_index("policy").loc["Oracle", "speedup_charged"]\n        df["frac_oracle_gap_closed"] = (df.speedup_charged - fb) / (orc - fb)\n    return df, sims\n\n\ndef write_selected(selected: dict, path, extra: dict | None = None) -> None:\n    out = dict(selected=selected, **(extra or {}))\n    Path(path).write_text(json.dumps(out, indent=1, ensure_ascii=False), encoding="utf-8")\n',
    'spec_loop': '"""Real greedy draft-and-verify loop, batch size 1, KV caches for both models (PLAN T9).\n\nThe *same* policy classes as the simulator run live: ``LiveView`` exposes ``feat`` / ``token`` with\nthe simulator\'s index convention (absolute output positions) and the same causality limit\n(``p + g - 1 + lookahead``). Draft features are computed from the draft\'s actual logits\n(conditioned on its own drafted tokens).\n\nRound at committed output length p (pending token = last committed token, not yet in the caches):\n  1. draft: feed every committed token missing from the draft cache, then draft greedily; after\n     each drafted token ask the policy (g >= 1). A lookahead-1 policy needs q_{p+g}, i.e. one more\n     draft forward; if the policy continues, that forward produced the next drafted token.\n  2. verify: one target forward on [pending, d_1..d_g]; accept the leading matches a; the target\n     argmax at position a is the correction / bonus token.\n  3. crop both caches to the committed length minus one (``crop_to``; transformers 5.22 ``crop``\n     takes the number of tokens to *remove*).\nOutput = committed tokens truncated after the first eos or at ``max_new``.\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom .simulator import GAMMA_MAX, CausalityError, Policy\n\nFEATURE_NAMES = ("H", "top1", "top2", "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont")\n\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef crop_to(cache, length: int) -> None:\n    """Crop a DynamicCache to ``length`` tokens (no-op if already shorter).\n\n    transformers 5.22: ``DynamicCache.crop(n)`` removes ``-n`` tokens for negative n and rejects positive n\n    (older versions took the final length). The result is checked so an API change fails loudly\n    instead of silently corrupting the cache."""\n    cur = cache.get_seq_length()\n    if cur > length:\n        cache.crop(-(cur - length))\n        got = cache.get_seq_length()\n        if got != length:\n            raise RuntimeError(f"cache crop failed: expected length {length}, got {got} (transformers API change?)")\n\n\nclass NoisyDraft:\n    """CPU smoke-test helper: wraps a causal LM and adds deterministic Gaussian noise to its logits,\n    seeded by the token that precedes each predicted position. Teacher-forced and live calls therefore\n    see the same draft distribution for the same context, while draft and target now disagree, so the\n    rejection / cache-crop path of the real loop is exercised. Not used on Kaggle."""\n\n    def __init__(self, model, sigma: float = 3.0, seed: int = 1234):\n        self.model, self.sigma, self.seed = model, float(sigma), int(seed)\n\n    def __getattr__(self, name):\n        return getattr(self.model, name)\n\n    def __call__(self, input_ids=None, **kw):\n        torch = _torch()\n        out = self.model(input_ids=input_ids, **kw)\n        V = out.logits.shape[-1]\n        prev = input_ids[0].tolist()\n        noise = torch.stack([torch.randn(V, generator=torch.Generator().manual_seed(self.seed * 1_000_003 + int(t)))\n                             for t in prev]).to(out.logits.device, out.logits.dtype)\n        out.logits = out.logits + self.sigma * noise[None]\n        return out\n\n\ndef _sync():\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\nclass LiveView:\n    """Causality-checked view of the live decoding state (same interface as ``TraceView``)."""\n\n    def __init__(self):\n        self.out: list[int] = []        # committed output tokens\n        self.draft: dict[int, int] = {}  # absolute idx -> drafted token (this round, incl. lookahead argmax)\n        self.feats: dict[int, dict] = {}  # absolute idx -> features of q_idx\n        self.p = 0\n        self.limit = -1\n        self.is_oracle = False\n        self.n = 10 ** 9\n\n    def _check(self, idx):\n        if idx > self.limit:\n            raise CausalityError(f"live read index {idx} > limit {self.limit} (p={self.p})")\n\n    def feat(self, name, idx):\n        self._check(idx)\n        if name == "draft_argmax":\n            return self.draft[idx]\n        if idx not in self.feats:\n            raise CausalityError(f"feature {name}[{idx}] not computed live (committed by the target)")\n        return self.feats[idx][name]\n\n    def token(self, idx):\n        self._check(idx)\n        return self.out[idx] if idx < self.p else self.draft[idx]\n\n    def gold(self, name, idx):\n        raise CausalityError("oracle policies cannot run live")\n\n\n@dataclass\nclass LoopResult:\n    out_ids: list\n    rounds: list = field(default_factory=list)  # dicts: p, g, accepted, extra_lookahead, t_draft, t_verify\n    wall: float = 0.0\n    n_draft_forwards: int = 0\n    n_target_forwards: int = 0\n\n\nclass SpecDecoder:\n    def __init__(self, target, draft, masks, eos_ids, device: str = "cuda:0", gamma_max: int = GAMMA_MAX):\n        torch = _torch()\n        from .gpu_traces import _mask_matrix\n        self.target, self.draft, self.device = target, draft, device\n        self.eos = set(int(i) for i in eos_ids)\n        self.gamma_max = gamma_max\n        V = draft.get_output_embeddings().weight.shape[0]\n        self.M = _mask_matrix(masks, V, device)            # (V, 5): space, punct, eos, mark, cont\n        self.torch = torch\n\n    # ---- model calls ---------------------------------------------------- #\n    def _fwd(self, model, ids, cache):\n        torch = self.torch\n        x = torch.tensor([ids], device=self.device)\n        with torch.no_grad():\n            return model(input_ids=x, past_key_values=cache, use_cache=True).logits[0].float()\n\n    def _features(self, logits_row) -> tuple[int, dict]:\n        torch = self.torch\n        lp = torch.log_softmax(logits_row, -1)\n        p = lp.exp()\n        top = p.topk(2)\n        P = p @ self.M.to(p.device)\n        H = -(p * lp).sum()\n        vals = torch.stack([H, top.values[0], top.values[1], P[0], P[1], P[2], P[0] + P[1] + P[2], P[3],\n                            P[4]]).cpu().tolist()\n        f = dict(zip(FEATURE_NAMES, vals))\n        return int(top.indices[0].item()), f\n\n    # ---- decoding --------------------------------------------------------- #\n    def decode(self, prompt_ids: list[int], policy: Policy, max_new: int, record: bool = True) -> LoopResult:\n        from transformers import DynamicCache\n        if policy.is_oracle:\n            raise ValueError("oracle policies cannot run live")\n        tc, dc = DynamicCache(), DynamicCache()\n        la = int(policy.lookahead)\n        res = LoopResult(out_ids=[])\n        _sync()\n        t_start = time.perf_counter()\n        # prefill: target on prompt[:-1], draft on prompt[:-1]; pending token = prompt[-1]\n        seq = list(prompt_ids)\n        if len(seq) > 1:\n            self._fwd(self.target, seq[:-1], tc)\n            self._fwd(self.draft, seq[:-1], dc)\n            res.n_target_forwards += 1\n            res.n_draft_forwards += 1\n        view = LiveView()\n        policy.reset(view)\n        n_prompt = len(prompt_ids)\n        done = False\n        while not done:\n            p = len(seq) - n_prompt          # committed output length\n            view.p, view.out = p, seq[n_prompt:]\n            view.draft, view.feats = {}, {}\n            cap = min(self.gamma_max, max_new - p)\n            _sync()\n            t0 = time.perf_counter()\n            drafted: list[int] = []\n            extra = False\n            g = 0\n            if cap > 0 and not policy.initial_stop(view, p):\n                # draft forward on all committed tokens missing from the draft cache -> q_p\n                missing = seq[dc.get_seq_length():]\n                lg = self._fwd(self.draft, missing, dc)[-1]\n                res.n_draft_forwards += 1\n                tok, f = self._features(lg)\n                view.draft[p], view.feats[p] = tok, f\n                drafted.append(tok)\n                g = 1\n                while g < cap:\n                    if la:   # q_{p+g}: feed the last drafted token\n                        lg = self._fwd(self.draft, [drafted[-1]], dc)[-1]\n                        res.n_draft_forwards += 1\n                        tok, f = self._features(lg)\n                        view.draft[p + g], view.feats[p + g] = tok, f\n                    view.limit = p + g - 1 + la\n                    if policy.stop(view, p, g):\n                        extra = bool(la)\n                        break\n                    if not la:\n                        lg = self._fwd(self.draft, [drafted[-1]], dc)[-1]\n                        res.n_draft_forwards += 1\n                        tok, f = self._features(lg)\n                        view.draft[p + g], view.feats[p + g] = tok, f\n                    drafted.append(view.draft[p + g])\n                    g += 1\n            _sync()\n            t1 = time.perf_counter()\n            # verify [pending] + drafted\n            lg = self._fwd(self.target, [seq[-1]] + drafted, tc)\n            res.n_target_forwards += 1\n            am = lg.argmax(-1).tolist()\n            a = 0\n            while a < g and am[a] == drafted[a]:\n                a += 1\n            new = drafted[:a] + [am[a]]\n            _sync()\n            t2 = time.perf_counter()\n            seq += new\n            L = len(seq)\n            crop_to(tc, L - 1)\n            crop_to(dc, L - 1)\n            if record:\n                res.rounds.append(dict(p=p, g=g, accepted=a, extra_lookahead=extra, t_draft=t1 - t0, t_verify=t2 - t1))\n            out = seq[n_prompt:]\n            if any(t in self.eos for t in new) or len(out) >= max_new:\n                done = True\n        _sync()\n        res.wall = time.perf_counter() - t_start\n        out = seq[n_prompt:]\n        cut = next((i + 1 for i, t in enumerate(out) if t in self.eos), len(out))\n        res.out_ids = out[:min(cut, max_new)]\n        return res\n\n\ndef first_divergence(a: list[int], b: list[int]) -> int | None:\n    for i, (x, y) in enumerate(zip(a, b)):\n        if x != y:\n            return i\n    return None if len(a) == len(b) else min(len(a), len(b))\n\n\ndef target_margin_at(target, prompt_ids, prefix_out, device) -> dict:\n    """Target top-1 vs top-2 logit margin for the next token after prompt + prefix_out (fp16 near-tie check)."""\n    torch = _torch()\n    x = torch.tensor([list(prompt_ids) + list(prefix_out)], device=device)\n    with torch.no_grad():\n        lg = target(x).logits[0, -1].float()\n    v, i = lg.topk(2)\n    return dict(top1=int(i[0]), top2=int(i[1]), margin=float(v[0] - v[1]))\n\n\n# --------------------------------------------------------------------------- #\n# notebook 03 entry point (also called by scripts/smoke_cpu.py --part 03)\n# --------------------------------------------------------------------------- #\n\n# MorphEntropy runs in its constrained form: the selected config has max_ext=0 and is identical to Entropy\n# (user decision 2026-11-02).\nLIVE_FAMILIES = ["NoDraft", "FixedK", "Entropy", "WordBoundarySoft", "Hybrid", "MorphEntropy_ext1"]\n\nDEFAULT_CONFIG_03 = dict(\n    N_SENT=100, REPEATS=3, MAX_NEW=256, SMOKE=False,\n    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",\n    DRAFT="meta-llama/Llama-3.2-1B-Instruct",\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    OUT_ROOT="/kaggle/working",\n    GIT_HASH="unknown",\n    POLICIES=LIVE_FAMILIES,\n    SELECTED=None,          # {family: params} from results/phase1/selected_params.json\n    PREFIX_MODEL_STATE=None,  # PrefixBoundaryModel state dict (decoded from the embedded gzip+base64)\n    IDS=None,               # devtest sentence ids (results/phase1/real_loop_ids.json)\n)\n\n\ndef _append_jsonl(path, rec):\n    with open(path, "a", encoding="utf-8") as fh:\n        fh.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n\n\ndef notebook03_main(cfg: dict) -> dict:\n    """PLAN T9 steps 1-8 on the GPU (or CPU with SMOKE=True: draft model as both target and draft)."""\n    import shutil\n    from pathlib import Path\n    from . import gpu_traces as G\n    from .morph import PrefixBoundaryModel\n    from .policies import TokenTable\n    from .traces import save_traces\n    from .tuning import Context, make_policy\n    from .vocab import build_masks\n    torch = _torch()\n    cfg = {**DEFAULT_CONFIG_03, **cfg}\n    out_dir = Path(cfg["OUT_ROOT"]) / "real_loop"\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    # ---- models ---- #\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        tok = G.load_tokenizer(cfg["DRAFT"])\n        draft = target = G.load_draft(cfg["DRAFT"], dtype, device)\n        if cfg.get("SMOKE_DRAFT_NOISE"):          # exercise rejections + cache cropping on CPU\n            draft = NoisyDraft(target, cfg["SMOKE_DRAFT_NOISE"])\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        device, dtype = "cuda:0", torch.float16\n        tok = G.load_tokenizer(cfg["TARGET"])\n        target = G.load_target(cfg["TARGET"], dtype, max_memory=cfg["MAX_MEMORY"])\n        draft = G.load_draft(cfg["DRAFT"], dtype, device)\n        G.check_draft_target_compat(draft, target, G.load_tokenizer(cfg["DRAFT"]), tok, "परीक्षण")\n    eos_ids = G.get_eos_ids(target, tok)\n    masks = build_masks(tok, eos_ids)\n    ctx = Context(newword_mask=masks.starts_newword, table=TokenTable.from_tokenizer(tok, masks),\n                  prefix_model=PrefixBoundaryModel.from_state(cfg["PREFIX_MODEL_STATE"]))\n    sel = cfg["SELECTED"]\n    policies = {f: make_policy(f, sel.get(f, {}), ctx) for f in cfg["POLICIES"]}\n    print("policies:", policies)\n    dec = SpecDecoder(target, draft, masks, eos_ids, device)\n\n    # ---- sentences ---- #\n    rows = {r[0]: r for r in G.load_flores_pairs("devtest")}\n    ids = list(cfg["IDS"])[: cfg["N_SENT"]]\n    prompts = {i: tok(G.build_prompt(tok, rows[i][1]), add_special_tokens=False)["input_ids"] for i in ids}\n\n    # ---- unbatched greedy reference (B9) ---- #\n    ref_path = out_dir / "greedy_unbatched.jsonl"\n    refs = {r["id"]: r for r in G.read_jsonl(ref_path)}\n    for i in ids:\n        if i not in refs:\n            o = G.generate_unbatched(target, tok, prompts[i], cfg["MAX_NEW"], eos_ids, device)\n            refs[i] = dict(id=i, src=rows[i][1], ref=rows[i][2], prompt_ids=prompts[i], out_ids=o,\n                           out_text=tok.decode(o, skip_special_tokens=True).strip())\n            _append_jsonl(ref_path, refs[i])\n    print(f"unbatched greedy references: {len(refs)}")\n\n    # ---- warm-up ---- #\n    for f, pol in policies.items():\n        dec.decode(prompts[ids[0]], pol, min(32, cfg["MAX_NEW"]), record=False)\n\n    # ---- timed runs (resumable) ---- #\n    res_path = out_dir / "loop_results.jsonl"\n    done = {(r["rep"], r["id"], r["policy"]) for r in G.read_jsonl(res_path)}\n    t_first = time.perf_counter()\n    n_new = 0\n    for rep in range(cfg["REPEATS"]):\n        for si, i in enumerate(ids):\n            for f, pol in policies.items():\n                if (rep, i, f) in done:\n                    continue\n                r = dec.decode(prompts[i], pol, cfg["MAX_NEW"])\n                lossless = r.out_ids == refs[i]["out_ids"]\n                _append_jsonl(res_path, dict(rep=rep, id=i, policy=f, params=sel.get(f, {}), wall=r.wall,\n                                             n_out=len(r.out_ids), out_ids=r.out_ids, lossless=lossless,\n                                             first_divergence=None if lossless else first_divergence(r.out_ids, refs[i]["out_ids"]),\n                                             n_draft_forwards=r.n_draft_forwards,\n                                             n_target_forwards=r.n_target_forwards, rounds=r.rounds))\n                n_new += 1\n            if rep == 0 and si == 4 and n_new:\n                per = (time.perf_counter() - t_first) / 5\n                est = per * len(ids) * cfg["REPEATS"] / 3600\n                print(f"ESTIMATE: {per:.1f}s per sentence (all policies) -> about {est:.1f} h for "\n                      f"N_SENT={len(ids)} x REPEATS={cfg[\'REPEATS\']}. Lower N_SENT / REPEATS if that is too long.")\n        print(f"repeat {rep} done")\n\n    # ---- losslessness details ---- #\n    results = G.read_jsonl(res_path)\n    div = []\n    for r in results:\n        if r["rep"] == 0 and not r["lossless"]:\n            pos = r["first_divergence"]\n            ref = refs[r["id"]]["out_ids"]\n            m = target_margin_at(target, prompts[r["id"]], ref[:pos], device) if pos is not None and pos < len(ref) else {}\n            div.append(dict(id=r["id"], policy=r["policy"], pos=pos, ref_token=ref[pos] if pos is not None and pos < len(ref) else None,\n                            loop_token=r["out_ids"][pos] if pos is not None and pos < len(r["out_ids"]) else None, **m))\n    (out_dir / "divergences.json").write_text(json.dumps(div, indent=1))\n    n_loss = sum(r["lossless"] for r in results)\n    print(f"losslessness: {n_loss} / {len(results)} decodes identical to unbatched greedy; {len(div)} divergences logged (rep 0)")\n\n    # ---- teacher-forced traces of the unbatched greedy outputs (for notebook 04) ---- #\n    recs = [refs[i] for i in ids]\n    dfe = G.run_draft_pass(draft, recs, masks, out_dir, "real100", device)\n    tfe = G.run_target_pass(target, recs, dfe, out_dir, "real100", device)\n    traces = G.assemble_traces(recs, dfe, tfe)\n    save_traces(out_dir / "traces_real100.npz", traces, dict(split="devtest-real100", eos_ids=eos_ids,\n                                                              git_hash=cfg["GIT_HASH"], max_new=cfg["MAX_NEW"]))\n\n    info = G.run_info(cfg["GIT_HASH"], n_sent=len(ids), repeats=cfg["REPEATS"], policies=list(policies),\n                      selected=sel, smoke=cfg["SMOKE"], target=cfg["TARGET"] if not cfg["SMOKE"] else cfg["DRAFT"],\n                      draft=cfg["DRAFT"], max_memory={str(k): v for k, v in cfg["MAX_MEMORY"].items()})\n    (out_dir / "run_info.json").write_text(json.dumps(info, indent=1, ensure_ascii=False))\n    zp = G.zip_dir(out_dir, Path(cfg["OUT_ROOT"]) / "real_loop.zip",\n                   include=["greedy_unbatched.jsonl", "loop_results.jsonl", "divergences.json",\n                            "traces_real100.npz", "run_info.json"])\n    print("\\n" + "=" * 70)\n    print("DONE. Download  real_loop.zip  from the notebook\'s Output tab")\n    print("and unzip it into  data/real_loop/  in the repo (files land directly in that folder).")\n    print("=" * 70)\n    return dict(zip=zp, n_lossless=n_loss, n_results=len(results), divergences=div)\n',
}
for _n, _s in LGMTP_SOURCES.items():
    with open(os.path.join(PKG, _n + '.py'), 'w', encoding='utf-8') as _f:
        _f.write(_s)
sys.path.insert(0, os.path.dirname(PKG))
for _m in [m for m in list(sys.modules) if m == 'lgmtp' or m.startswith('lgmtp.')]:
    del sys.modules[_m]
import lgmtp
LGMTP_BUILD = 'git=a253e1ed509c-dirty src+artifacts_sha256=0f135c92b058'
print('lgmtp build', LGMTP_BUILD)

## CONFIG

In [ ]:
ARTIFACT_DIR = None   # only needed if the notebook says an artifacts dataset is required
CONFIG = dict(
    N_SENT=100,          # sentences (taken in order from the fixed id list)
    REPEATS=3,          # timed repeats per sentence and policy (median is reported)
    MAX_NEW=256,
    SMOKE=False,
    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",
    DRAFT="meta-llama/Llama-3.2-1B-Instruct",
    MAX_MEMORY={0: "8GiB", 1: "10GiB"},
    OUT_ROOT="/kaggle/working",
    GIT_HASH=LGMTP_BUILD,
)
RESUME_FROM = None    # e.g. "/kaggle/input/<earlier-version-output>" to continue a dead run

In [ ]:
import json
SELECTED = json.loads('{"FixedK": {"k": 2}, "Entropy": {"lam": -0.683, "beta": 4.0}, "NoDraft": {}, "WordBoundarySoft": {"tau": 0.05, "min_g": 2}, "WordBoundaryHard": {"min_g": 2}, "AksharaSafeEntropy": {"lam": -0.746, "beta": 4.0, "tau_m": 0.7, "max_ext": 1}, "Hybrid": {"lam": -0.621, "beta": 2.0, "tau_w": 0.2, "max_ext": 0}, "MorphEntropy": {"lam": -0.683, "beta": 4.0, "tau_morph": 0.1, "use_akshara": false, "tau_m": 0.5, "max_ext": 0}, "MorphOnly": {"tau_morph": 0.05, "min_g": 2}, "Oracle": {}, "GoldWordOracle": {"min_g": 1}, "Hybrid_ext1": {"lam": -0.683, "beta": 4.0, "tau_w": 0.2, "max_ext": 1}, "MorphEntropy_ext1": {"lam": -0.621, "beta": 4.0, "tau_morph": 0.1, "use_akshara": false, "tau_m": 0.5, "max_ext": 1}}')
IDS = [2, 5, 7, 15, 20, 27, 31, 37, 48, 69, 72, 76, 84, 118, 132, 161, 165, 228, 247, 253, 258, 265, 282, 284, 305, 320, 329, 337, 354, 367, 369, 379, 381, 384, 392, 402, 416, 444, 457, 465, 467, 479, 503, 504, 505, 513, 515, 520, 521, 526, 561, 566, 568, 582, 587, 592, 593, 599, 621, 627, 628, 632, 642, 664, 668, 670, 676, 686, 687, 707, 709, 717, 719, 735, 749, 761, 762, 776, 778, 798, 804, 816, 817, 829, 842, 843, 855, 871, 878, 893, 899, 922, 924, 949, 960, 963, 965, 982, 1006, 1009]
import base64, gzip, io, pickle
_PM_B64 = '...'
PREFIX_MODEL_STATE = pickle.load(gzip.open(io.BytesIO(base64.b64decode(_PM_B64)), 'rb'))
print('prefix model levels:', {k: len(v) for k, v in PREFIX_MODEL_STATE['tot'].items()})

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with Kaggle secret HF_TOKEN")
except Exception as e:
    print(f"Kaggle secret HF_TOKEN not available ({e})")
    login()
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import shutil
from pathlib import Path
if RESUME_FROM:   # copy partial outputs of a dead run back into place
    dst = Path(CONFIG["OUT_ROOT"]) / "real_loop"; dst.mkdir(parents=True, exist_ok=True)
    for f in Path(RESUME_FROM).rglob("*"):
        if f.is_file() and f.parent.name == "real_loop" and not (dst / f.name).exists():
            shutil.copy2(f, dst / f.name); print("resume: copied", f)
from lgmtp.spec_loop import notebook03_main
RESULT = notebook03_main(dict(CONFIG, SELECTED=SELECTED, IDS=IDS, PREFIX_MODEL_STATE=PREFIX_MODEL_STATE))